In [2]:
# Cell 1 — Environment Verification
# Purpose: Confirm Python version and availability of core libraries we will use.
# This is our baseline checkpoint before any installation or data work begins.

import sys
import importlib

print("=" * 50)
print("ENVIRONMENT VERIFICATION")
print("=" * 50)

# Python version
print(f"\nPython version: {sys.version}")

# Libraries to check: (import_name, pip_name)
libraries_to_check = [
    ("requests", "requests"),
    ("bs4", "beautifulsoup4"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("tqdm", "tqdm"),
    ("pyarrow", "pyarrow"),
]

print("\nLibrary availability:")
print("-" * 40)

all_good = True
for import_name, pip_name in libraries_to_check:
    spec = importlib.util.find_spec(import_name)
    if spec is not None:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "version unknown")
        print(f"  ✓  {pip_name:<20} {version}")
    else:
        print(f"  ✗  {pip_name:<20} NOT FOUND")
        all_good = False

print("-" * 40)
if all_good:
    print("\n✓ All core libraries present. Ready to proceed.")
else:
    print("\n✗ Some libraries are missing. We will install them next.")

ENVIRONMENT VERIFICATION

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Library availability:
----------------------------------------
  ✓  requests             2.32.4
  ✓  beautifulsoup4       4.13.5
  ✓  pandas               2.2.2
  ✓  numpy                2.0.2
  ✓  tqdm                 4.67.3
  ✓  pyarrow              18.1.0
----------------------------------------

✓ All core libraries present. Ready to proceed.


In [3]:
# Cell 2 — Connectivity Check
# Purpose: Verify that Colab can reach the Wikimedia clickstream dump server.
# We also inspect what months of data are currently available.

import requests

print("=" * 50)
print("CONNECTIVITY CHECK — WIKIMEDIA DUMP SERVER")
print("=" * 50)

CLICKSTREAM_BASE_URL = "https://dumps.wikimedia.org/other/clickstream/"

print(f"\nTarget URL: {CLICKSTREAM_BASE_URL}")
print("Sending request...\n")

try:
    response = requests.get(CLICKSTREAM_BASE_URL, timeout=15)
    print(f"Status code : {response.status_code}")
    print(f"Response size: {len(response.text)} characters")

    if response.status_code == 200:
        print("\n✓ Server is reachable.")

        # Parse the directory listing to find available months
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(response.text, "html.parser")

        # Each available month is a link ending in /
        months = [
            a.get_text(strip=True)
            for a in soup.find_all("a")
            if a.get_text(strip=True).startswith("20")  # e.g. 2024-03/
        ]

        print(f"\nAvailable monthly dumps ({len(months)} found):")
        print("-" * 40)
        for m in months:
            print(f"  {m}")

        print("-" * 40)
        print(f"\nMost recent month: {months[-1] if months else 'unknown'}")

    else:
        print(f"\n✗ Unexpected status code: {response.status_code}")

except requests.exceptions.Timeout:
    print("✗ Request timed out. Check Colab's network or try again.")
except requests.exceptions.ConnectionError as e:
    print(f"✗ Connection error: {e}")

CONNECTIVITY CHECK — WIKIMEDIA DUMP SERVER

Target URL: https://dumps.wikimedia.org/other/clickstream/
Sending request...

Status code : 200
Response size: 11926 characters

✓ Server is reachable.

Available monthly dumps (103 found):
----------------------------------------
  2017-11/
  2017-12/
  2018-01/
  2018-02/
  2018-03/
  2018-04/
  2018-05/
  2018-06/
  2018-07/
  2018-08/
  2018-09/
  2018-10/
  2018-11/
  2018-12/
  2019-01/
  2019-02/
  2019-03/
  2019-04/
  2019-05/
  2019-06/
  2019-07/
  2019-08/
  2019-09/
  2019-10/
  2019-11/
  2019-12/
  2020-01/
  2020-02/
  2020-03/
  2020-04/
  2020-05/
  2020-06/
  2020-07/
  2020-08/
  2020-09/
  2020-10/
  2020-11/
  2020-12/
  2021-01/
  2021-02/
  2021-03/
  2021-04/
  2021-05/
  2021-06/
  2021-07/
  2021-08/
  2021-09/
  2021-10/
  2021-11/
  2021-12/
  2022-01/
  2022-02/
  2022-03/
  2022-04/
  2022-05/
  2022-06/
  2022-07/
  2022-08/
  2022-09/
  2022-10/
  2022-11/
  2022-12/
  2023-01/
  2023-02/
  2023-03/
  2023-04

In [4]:
# Cell 3 — Inspect a Monthly Dump Folder
# Purpose: Look inside the most recent monthly folder and identify
# the exact filename we need for English Wikipedia clickstream data.

import requests
from bs4 import BeautifulSoup

# We'll inspect the most recent month
TARGET_MONTH = "2026-05"
FOLDER_URL = f"https://dumps.wikimedia.org/other/clickstream/{TARGET_MONTH}/"

print("=" * 55)
print(f"INSPECTING DUMP FOLDER: {TARGET_MONTH}")
print("=" * 55)
print(f"\nURL: {FOLDER_URL}\n")

response = requests.get(FOLDER_URL, timeout=15)

if response.status_code != 200:
    print(f"✗ Failed to reach folder. Status: {response.status_code}")
else:
    soup = BeautifulSoup(response.text, "html.parser")

    # Extract all file links (filter out parent directory link)
    files = [
        a.get_text(strip=True)
        for a in soup.find_all("a")
        if a.get_text(strip=True).endswith(".gz")
    ]

    print(f"Files available in {TARGET_MONTH}/ ({len(files)} total):\n")
    print("-" * 55)
    for f in files:
        print(f"  {f}")
    print("-" * 55)

    # Identify the English Wikipedia file specifically
    english_files = [f for f in files if "_en_" in f or "-en-" in f]
    print(f"\nEnglish Wikipedia files:")
    for f in english_files:
        print(f"  → {f}")

    if english_files:
        print(f"\n✓ Target file identified: {english_files[0]}")
    else:
        print("\n✗ Could not auto-detect English file. Review list above manually.")

INSPECTING DUMP FOLDER: 2026-05

URL: https://dumps.wikimedia.org/other/clickstream/2026-05/

Files available in 2026-05/ (40 total):

-------------------------------------------------------
  clickstream-arwiki-2026-05.tsv.gz
  clickstream-azwiki-2026-05.tsv.gz
  clickstream-bgwiki-2026-05.tsv.gz
  clickstream-bnwiki-2026-05.tsv.gz
  clickstream-cawiki-2026-05.tsv.gz
  clickstream-cswiki-2026-05.tsv.gz
  clickstream-dawiki-2026-05.tsv.gz
  clickstream-dewiki-2026-05.tsv.gz
  clickstream-elwiki-2026-05.tsv.gz
  clickstream-enwiki-2026-05.tsv.gz
  clickstream-eswiki-2026-05.tsv.gz
  clickstream-fawiki-2026-05.tsv.gz
  clickstream-fiwiki-2026-05.tsv.gz
  clickstream-frwiki-2026-05.tsv.gz
  clickstream-hewiki-2026-05.tsv.gz
  clickstream-hiwiki-2026-05.tsv.gz
  clickstream-hrwiki-2026-05.tsv.gz
  clickstream-huwiki-2026-05.tsv.gz
  clickstream-idwiki-2026-05.tsv.gz
  clickstream-itwiki-2026-05.tsv.gz
  clickstream-jawiki-2026-05.tsv.gz
  clickstream-kowiki-2026-05.tsv.gz
  clickstream-mlw

In [5]:
# Cell 4 — Confirm English File URL and File Size
# Purpose: Fix detection logic, construct the full download URL,
# and do a HEAD request to confirm the file exists and check its size.

import requests

TARGET_MONTH = "2026-05"
FILENAME = f"clickstream-enwiki-{TARGET_MONTH}.tsv.gz"
DOWNLOAD_URL = f"https://dumps.wikimedia.org/other/clickstream/{TARGET_MONTH}/{FILENAME}"

print("=" * 55)
print("ENGLISH FILE URL VERIFICATION")
print("=" * 55)
print(f"\nFilename : {FILENAME}")
print(f"Full URL : {DOWNLOAD_URL}\n")

# HEAD request — fetches metadata only, does not download the file
response = requests.head(DOWNLOAD_URL, timeout=15, allow_redirects=True)

print(f"Status code : {response.status_code}")

if response.status_code == 200:
    # File size in bytes → convert to MB
    size_bytes = int(response.headers.get("Content-Length", 0))
    size_mb = size_bytes / (1024 * 1024)
    content_type = response.headers.get("Content-Type", "unknown")

    print(f"Content-Type: {content_type}")
    print(f"File size   : {size_bytes:,} bytes ({size_mb:.1f} MB)")
    print(f"\n✓ File confirmed. Ready to download {size_mb:.1f} MB.")
else:
    print(f"\n✗ File not reachable. Status: {response.status_code}")
    print("Double-check the filename manually from the listing above.")

ENGLISH FILE URL VERIFICATION

Filename : clickstream-enwiki-2026-05.tsv.gz
Full URL : https://dumps.wikimedia.org/other/clickstream/2026-05/clickstream-enwiki-2026-05.tsv.gz

Status code : 200
Content-Type: application/octet-stream
File size   : 508,147,113 bytes (484.6 MB)

✓ File confirmed. Ready to download 484.6 MB.


In [6]:
# Cell 5 — Streaming Download with Progress Bar
# Purpose: Download the compressed clickstream file to Colab local disk.
# Uses chunk-based streaming to keep memory usage low and stable.

import requests
import os
from tqdm import tqdm

TARGET_MONTH = "2026-05"
FILENAME = f"clickstream-enwiki-{TARGET_MONTH}.tsv.gz"
DOWNLOAD_URL = f"https://dumps.wikimedia.org/other/clickstream/{TARGET_MONTH}/{FILENAME}"

# Store in /content/ which is Colab's writable local disk
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(DATA_DIR, FILENAME)

print("=" * 55)
print("STREAMING DOWNLOAD")
print("=" * 55)
print(f"\nSource : {DOWNLOAD_URL}")
print(f"Dest   : {OUTPUT_PATH}\n")

CHUNK_SIZE = 1024 * 1024  # 1 MB chunks

response = requests.get(DOWNLOAD_URL, stream=True, timeout=60)
total_bytes = int(response.headers.get("Content-Length", 0))

if response.status_code != 200:
    print(f"✗ Download failed. Status: {response.status_code}")
else:
    with open(OUTPUT_PATH, "wb") as f:
        with tqdm(
            total=total_bytes,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=FILENAME,
            ncols=80
        ) as progress:
            for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
                    progress.update(len(chunk))

    # Verify the file landed correctly
    actual_size = os.path.getsize(OUTPUT_PATH)
    print(f"\nExpected : {total_bytes:,} bytes")
    print(f"Received : {actual_size:,} bytes")

    if actual_size == total_bytes:
        print("✓ Download complete. File integrity confirmed.")
    else:
        print("✗ Size mismatch — file may be corrupted. Do not proceed.")

STREAMING DOWNLOAD

Source : https://dumps.wikimedia.org/other/clickstream/2026-05/clickstream-enwiki-2026-05.tsv.gz
Dest   : /content/data/clickstream-enwiki-2026-05.tsv.gz



clickstream-enwiki-2026-05.tsv.gz: 100%|█████| 485M/485M [01:55<00:00, 4.39MB/s]


Expected : 508,147,113 bytes
Received : 508,147,113 bytes
✓ Download complete. File integrity confirmed.


In [7]:
# Cell 6 — Inspect Raw File Structure
# Purpose: Read the first 20 lines of the compressed file without
# fully decompressing it. Understand columns, delimiter, and data types.

import gzip
import os

DATA_DIR = "/content/data"
TARGET_MONTH = "2026-05"
FILENAME = f"clickstream-enwiki-{TARGET_MONTH}.tsv.gz"
FILE_PATH = os.path.join(DATA_DIR, FILENAME)

print("=" * 60)
print("RAW FILE INSPECTION (first 20 lines)")
print("=" * 60)

N_LINES = 20

with gzip.open(FILE_PATH, "rt", encoding="utf-8") as f:
    lines = [next(f) for _ in range(N_LINES)]

print(f"\nRaw lines as-is:\n")
print("-" * 60)
for i, line in enumerate(lines):
    print(f"[{i:02d}] {line}", end="")
print("-" * 60)

# Analyze delimiter
first_line = lines[0]
tab_count   = first_line.count("\t")
comma_count = first_line.count(",")
pipe_count  = first_line.count("|")

print(f"\nDelimiter analysis on line 0:")
print(f"  Tabs   : {tab_count}")
print(f"  Commas : {comma_count}")
print(f"  Pipes  : {pipe_count}")

# Infer number of columns
if tab_count > 0:
    cols = first_line.strip().split("\t")
    print(f"\nInferred delimiter : TAB")
    print(f"Inferred columns   : {len(cols)}")
    print(f"\nColumn values (line 0):")
    for i, col in enumerate(cols):
        print(f"  col[{i}] = '{col}'")

RAW FILE INSPECTION (first 20 lines)

Raw lines as-is:

------------------------------------------------------------
[00] other-empty	Main_Page	external	199920521
[01] Bagua	Miscellaneous_Symbols	link	13
[02] other-internal	Main_Page	external	12152035
[03] Bagua	Yin-yang-style_baguazhang	other	13
[04] other-empty	Hyphen-minus	external	5873607
[05] Baguazhang	Gun_(staff)	link	13
[06] other-search	.xxx	external	5447828
[07] Baguette	French_Parliament	link	13
[08] other-search	C._Joseph_Vijay	external	3408673
[09] Baguette	Marie_Blachère	link	13
[10] other-search	Obsession_(2025_film)	external	3245873
[11] Baguette	Pâté	link	13
[12] other-search	2026_Tamil_Nadu_Legislative_Assembly_election	external	2507241
[13] Baguiati	Lake_Town,_Kolkata	link	13
[14] other-empty	Neatsville,_Kentucky	external	2436428
[15] Baguim_do_Monte	Gondomar,_Portugal	link	13
[16] other-search	Murder_of_Dominic_Russo_and_Davion_Flanagan	external	2373437
[17] Baguio	Main_Page	other	13
[18] other-search	Michael_Jackso

In [8]:
# Cell 7 — Load and Filter Clickstream Data
# Purpose: Read the full compressed TSV into pandas.
# Apply column names, filter to Wikipedia-internal link clicks only,
# and report dataset statistics.

import gzip
import pandas as pd
import os

DATA_DIR = "/content/data"
TARGET_MONTH = "2026-05"
FILENAME = f"clickstream-enwiki-{TARGET_MONTH}.tsv.gz"
FILE_PATH = os.path.join(DATA_DIR, FILENAME)

print("=" * 55)
print("LOADING CLICKSTREAM DATA")
print("=" * 55)

# Column names per Wikimedia's official documentation
COL_NAMES = ["prev", "curr", "type", "n"]

print("\nReading compressed TSV into pandas...")

df_raw = pd.read_csv(
    FILE_PATH,
    sep="\t",
    names=COL_NAMES,
    compression="gzip",
    dtype={"prev": str, "curr": str, "type": str, "n": int},
)

print(f"✓ File loaded.")
print(f"\nRaw dataset shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

# Distribution of 'type' values
print(f"\nRow counts by type:")
print("-" * 35)
type_counts = df_raw["type"].value_counts()
for t, count in type_counts.items():
    pct = count / len(df_raw) * 100
    print(f"  {t:<15} {count:>10,}  ({pct:.1f}%)")
print("-" * 35)

# Filter to internal Wikipedia link clicks only
df_links = df_raw[df_raw["type"] == "link"].copy()
df_links.reset_index(drop=True, inplace=True)

print(f"\nAfter filtering to type='link':")
print(f"  Rows retained : {len(df_links):,}")
print(f"  Rows dropped  : {len(df_raw) - len(df_links):,}")

# Basic statistics on click counts
print(f"\nClick count (n) statistics:")
print("-" * 35)
stats = df_links["n"].describe()
for stat, val in stats.items():
    print(f"  {stat:<10} {val:>12,.1f}")
print("-" * 35)

print(f"\nSample of link-type rows:")
print(df_links.head(10).to_string(index=False))

LOADING CLICKSTREAM DATA

Reading compressed TSV into pandas...


ParserError: Error tokenizing data. C error: Expected 4 fields in line 529485, saw 5


In [ ]:
# Cell 8 — Reload with Error Tolerance + Inspect Bad Rows
# Purpose: Use on_bad_lines='warn' to skip malformed rows without crashing,
# then separately extract and inspect the bad lines to understand them.

import gzip
import pandas as pd
import os

DATA_DIR = "/content/data"
TARGET_MONTH = "2026-05"
FILENAME = f"clickstream-enwiki-{TARGET_MONTH}.tsv.gz"
FILE_PATH = os.path.join(DATA_DIR, FILENAME)

COL_NAMES = ["prev", "curr", "type", "n"]

print("=" * 55)
print("RELOADING WITH ERROR TOLERANCE")
print("=" * 55)

print("\nPass 1: Load with on_bad_lines='skip', count skipped rows...")

# Capture warnings to count bad lines
import warnings
import io

# Load skipping bad lines
df_raw = pd.read_csv(
    FILE_PATH,
    sep="\t",
    names=COL_NAMES,
    compression="gzip",
    dtype={"prev": str, "curr": str, "type": str, "n": str},
    on_bad_lines="skip",
    engine="python",
)

print(f"✓ Loaded successfully.")
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

# Now inspect a sample of raw lines near the reported bad line
print(f"\nPass 2: Manually read raw lines around line 529,485...")
print("-" * 55)

TARGET_LINE = 529485
WINDOW = 3  # lines before and after

with gzip.open(FILE_PATH, "rt", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        if TARGET_LINE - WINDOW <= i <= TARGET_LINE + WINDOW:
            tab_count = line.count("\t")
            print(f"[line {i:>7}] tabs={tab_count} | {repr(line[:120])}")
        if i > TARGET_LINE + WINDOW:
            break

print("-" * 55)

# Type distribution on what we did load
print(f"\nType distribution in loaded data:")
print(df_raw["type"].value_counts().to_string())

RELOADING WITH ERROR TOLERANCE

Pass 1: Load with on_bad_lines='skip', count skipped rows...


In [1]:
# Cell 9 — Stream-Filter to Link-Only TSV
# Purpose: Read the compressed file line by line, keep only type='link' rows,
# skip any malformed lines, write clean output to a smaller file.
# Memory usage stays flat — only one line in RAM at a time.

import gzip
import os
from tqdm import tqdm

DATA_DIR = "/content/data"
TARGET_MONTH = "2026-05"
INPUT_FILE  = os.path.join(DATA_DIR, f"clickstream-enwiki-{TARGET_MONTH}.tsv.gz")
OUTPUT_FILE = os.path.join(DATA_DIR, f"links-only-enwiki-{TARGET_MONTH}.tsv")

print("=" * 55)
print("STREAM FILTERING: keeping type='link' rows only")
print("=" * 55)
print(f"\nInput  : {INPUT_FILE}")
print(f"Output : {OUTPUT_FILE}\n")

total_lines  = 0
kept_lines   = 0
bad_lines    = 0

with gzip.open(INPUT_FILE, "rt", encoding="utf-8", errors="replace") as infile, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:

    # Write header to output file
    outfile.write("prev\tcurr\ttype\tn\n")

    for line in tqdm(infile, unit=" lines", ncols=70, mininterval=2.0):
        total_lines += 1
        parts = line.rstrip("\n").split("\t")

        # Skip malformed lines (not exactly 4 fields)
        if len(parts) != 4:
            bad_lines += 1
            continue

        prev, curr, link_type, n = parts

        # Keep only internal Wikipedia link clicks
        if link_type != "link":
            continue

        # Validate n is an integer
        if not n.strip().isdigit():
            bad_lines += 1
            continue

        outfile.write(line)
        kept_lines += 1

print(f"\n{'='*55}")
print(f"Total lines read : {total_lines:>12,}")
print(f"Bad/malformed    : {bad_lines:>12,}")
print(f"Kept (link type) : {kept_lines:>12,}")
print(f"Discarded        : {total_lines - kept_lines - bad_lines:>12,}")

out_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f"\nOutput file size : {out_size_mb:.1f} MB")
print(f"Output path      : {OUTPUT_FILE}")
print(f"\n✓ Done. Clean file ready for pandas.")


STREAM FILTERING: keeping type='link' rows only

Input  : /content/data/clickstream-enwiki-2026-05.tsv.gz
Output : /content/data/links-only-enwiki-2026-05.tsv



35419767 lines [01:21, 434650.37 lines/s]


Total lines read :   35,419,767
Bad/malformed    :            0
Kept (link type) :   22,468,830
Discarded        :   12,950,937

Output file size : 1047.4 MB
Output path      : /content/data/links-only-enwiki-2026-05.tsv

✓ Done. Clean file ready for pandas.


In [2]:
# Cell 10 — Load Filtered Data with Optimized Dtypes
# Purpose: Load the clean link-only TSV into pandas efficiently.
# Use category dtype for 'type' column, int32 for click counts.
# Report memory usage and basic statistics.

import pandas as pd
import os

DATA_DIR    = "/content/data"
TARGET_MONTH = "2026-05"
INPUT_FILE  = os.path.join(DATA_DIR, f"links-only-enwiki-{TARGET_MONTH}.tsv")

print("=" * 55)
print("LOADING FILTERED DATA INTO PANDAS")
print("=" * 55)
print(f"\nFile: {INPUT_FILE}")
print(f"Loading...\n")

df = pd.read_csv(
    INPUT_FILE,
    sep="\t",
    dtype={
        "prev": "string",    # pandas StringDtype (nullable, memory efficient)
        "curr": "string",    # pandas StringDtype
        "type": "category",  # only one unique value 'link' but good practice
        "n":    "int32",     # max click value fits easily in int32
    },
)

print(f"✓ Loaded successfully.\n")
print(f"Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")

# Memory usage
mem = df.memory_usage(deep=True)
total_mem_mb = mem.sum() / (1024 * 1024)
print(f"\nMemory usage per column:")
print("-" * 40)
for col in df.columns:
    col_mb = mem[col] / (1024 * 1024)
    print(f"  {col:<6} {col_mb:>8.1f} MB")
print("-" * 40)
print(f"  {'TOTAL':<6} {total_mem_mb:>8.1f} MB")

# Basic stats
print(f"\nClick count (n) statistics:")
print("-" * 40)
desc = df["n"].describe()
for stat, val in desc.items():
    print(f"  {stat:<10} {val:>12,.0f}")
print("-" * 40)

# Unique article counts
print(f"\nUnique source articles (prev) : {df['prev'].nunique():>10,}")
print(f"Unique target articles (curr) : {df['curr'].nunique():>10,}")

print(f"\nFirst 5 rows:")
print(df.head().to_string(index=False))

LOADING FILTERED DATA INTO PANDAS

File: /content/data/links-only-enwiki-2026-05.tsv
Loading...



ValueError: Integer column has NA values in column 3

In [3]:
# Cell 11 — Handle NA Values in Click Count Column
# Purpose: Load with nullable Int32 to tolerate NA values,
# inspect affected rows, then drop them and report final clean shape.

import pandas as pd
import os

DATA_DIR     = "/content/data"
TARGET_MONTH = "2026-05"
INPUT_FILE   = os.path.join(DATA_DIR, f"links-only-enwiki-{TARGET_MONTH}.tsv")

print("=" * 55)
print("LOADING WITH NULLABLE INTEGER — NA INSPECTION")
print("=" * 55)

df = pd.read_csv(
    INPUT_FILE,
    sep="\t",
    dtype={
        "prev": "string",
        "curr": "string",
        "type": "category",
        "n":    "Int32",     # Capital I — nullable integer, tolerates NA
    },
)

print(f"✓ Loaded: {df.shape[0]:,} rows\n")

# Count NAs per column
print("NA counts per column:")
print("-" * 35)
na_counts = df.isna().sum()
for col, count in na_counts.items():
    print(f"  {col:<6} {count:>10,}")
print("-" * 35)

# Inspect the NA rows in column n
na_rows = df[df["n"].isna()]
print(f"\nSample of rows where n is NA ({len(na_rows):,} total):")
print(na_rows.head(10).to_string(index=False))

# Drop NA rows
df.dropna(subset=["n"], inplace=True)
df["n"] = df["n"].astype("int32")  # downcast to int32 now that NAs are gone
df.reset_index(drop=True, inplace=True)

print(f"\nAfter dropping NA rows:")
print(f"  Rows remaining : {len(df):,}")

# Memory usage
total_mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"  Memory usage   : {total_mem_mb:.1f} MB")

print(f"\nFirst 5 clean rows:")
print(df.head().to_string(index=False))
print(f"\n✓ Clean DataFrame ready.")

LOADING WITH NULLABLE INTEGER — NA INSPECTION


ParserError: Error tokenizing data. C error: Expected 4 fields in line 3005872, saw 5


In [4]:
# Cell 12 — Re-Stream Filter with Tab Sanitization
# Purpose: Rebuild the clean TSV, this time reconstructing each output line
# from parsed fields rather than copying the raw line.
# This eliminates embedded tabs in article titles.

import gzip
import os
from tqdm import tqdm

DATA_DIR     = "/content/data"
TARGET_MONTH = "2026-05"
INPUT_FILE   = os.path.join(DATA_DIR, f"clickstream-enwiki-{TARGET_MONTH}.tsv.gz")
OUTPUT_FILE  = os.path.join(DATA_DIR, f"links-only-enwiki-{TARGET_MONTH}.tsv")

print("=" * 55)
print("RE-FILTERING WITH TAB SANITIZATION")
print("=" * 55)
print(f"\nInput  : {INPUT_FILE}")
print(f"Output : {OUTPUT_FILE}\n")

total_lines = 0
kept_lines  = 0
bad_lines   = 0
tab_fixed   = 0

with gzip.open(INPUT_FILE, "rt", encoding="utf-8", errors="replace") as infile, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:

    outfile.write("prev\tcurr\ttype\tn\n")

    for line in tqdm(infile, unit=" lines", ncols=70, mininterval=2.0):
        total_lines += 1
        parts = line.rstrip("\n").split("\t")

        # We expect exactly 4 fields
        if len(parts) < 4:
            bad_lines += 1
            continue

        # If more than 4 fields, article titles contain embedded tabs.
        # Structure is always: prev, curr, type, n
        # 'type' is always one of: link, external, other
        # 'n' is always the last field (integer)
        # So we take: last field = n, second-to-last = type,
        # everything between first and type = belongs to curr,
        # first field = prev
        if len(parts) > 4:
            n         = parts[-1]
            link_type = parts[-2]
            curr      = " ".join(parts[1:-2])  # collapse middle fields
            prev      = parts[0]
            tab_fixed += 1
        else:
            prev, curr, link_type, n = parts

        # Only keep internal Wikipedia link clicks
        if link_type != "link":
            continue

        # Validate n is a positive integer
        if not n.strip().isdigit():
            bad_lines += 1
            continue

        # Sanitize: remove any remaining tabs from article names
        prev = prev.replace("\t", " ").strip()
        curr = curr.replace("\t", " ").strip()

        # Reconstruct the line cleanly from parsed fields
        outfile.write(f"{prev}\t{curr}\t{link_type}\t{n}\n")
        kept_lines += 1

print(f"\n{'='*55}")
print(f"Total lines read  : {total_lines:>12,}")
print(f"Bad/malformed     : {bad_lines:>12,}")
print(f"Tab-fixed rows    : {tab_fixed:>12,}")
print(f"Kept (link type)  : {kept_lines:>12,}")
print(f"Discarded (other) : {total_lines - kept_lines - bad_lines:>12,}")

out_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f"\nOutput file size  : {out_size_mb:.1f} MB")
print(f"\n✓ Clean file written. Ready to load into pandas.")

RE-FILTERING WITH TAB SANITIZATION

Input  : /content/data/clickstream-enwiki-2026-05.tsv.gz
Output : /content/data/links-only-enwiki-2026-05.tsv



35419767 lines [01:35, 372625.93 lines/s]


Total lines read  :   35,419,767
Bad/malformed     :            0
Tab-fixed rows    :            0
Kept (link type)  :   22,468,830
Discarded (other) :   12,950,937

Output file size  : 1047.4 MB

✓ Clean file written. Ready to load into pandas.


In [5]:
# Cell 13 — Bulletproof Load: All Strings First, Then Convert
# Purpose: Load entire file as string dtype (never crashes on type issues),
# then inspect and convert each column deliberately.

import pandas as pd
import os

DATA_DIR     = "/content/data"
TARGET_MONTH = "2026-05"
INPUT_FILE   = os.path.join(DATA_DIR, f"links-only-enwiki-{TARGET_MONTH}.tsv")

print("=" * 55)
print("BULLETPROOF LOAD — ALL STRING DTYPES")
print("=" * 55)
print(f"\nFile: {INPUT_FILE}")
print(f"Loading all columns as str...\n")

df = pd.read_csv(
    INPUT_FILE,
    sep="\t",
    dtype=str,           # Everything as string — never crashes
    on_bad_lines="skip", # Skip any remaining bad lines
    engine="c",          # C engine is fast, safe with dtype=str
)

print(f"✓ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")

# Inspect each column
print(f"\nNA counts per column:")
for col in df.columns:
    print(f"  {col:<6}: {df[col].isna().sum():,} NAs")

# Inspect unexpected values in 'n' column before converting
print(f"\nUnique non-numeric values in 'n' column:")
non_numeric_n = df[~df["n"].str.strip().str.isdigit()]
print(f"  Count: {len(non_numeric_n):,}")
if len(non_numeric_n) > 0:
    print(non_numeric_n.head(10).to_string(index=False))

# Drop rows where n is not a clean integer
df = df[df["n"].str.strip().str.isdigit()].copy()

# Convert n to int32
df["n"] = df["n"].str.strip().astype("int32")
df["type"] = df["type"].astype("category")
df.reset_index(drop=True, inplace=True)

print(f"\nAfter type conversion and cleanup:")
print(f"  Rows : {len(df):,}")

mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"  RAM  : {mem_mb:.1f} MB")

print(f"\nSample:")
print(df.head(5).to_string(index=False))
print(f"\n✓ Clean DataFrame ready.")

BULLETPROOF LOAD — ALL STRING DTYPES

File: /content/data/links-only-enwiki-2026-05.tsv
Loading all columns as str...

✓ Loaded: 22,273,328 rows × 4 columns

Columns: ['prev', 'curr', 'type', 'n']

NA counts per column:
  prev  : 29 NAs
  curr  : 41 NAs
  type  : 0 NAs
  n     : 21 NAs

Unique non-numeric values in 'n' column:


TypeError: bad operand type for unary ~: 'float'

In [6]:
# Cell 14 — Final Cleanup: Drop NAs and Convert Types
# Purpose: Drop the small number of NA rows, convert n to int32,
# and confirm our final clean DataFrame.

import pandas as pd

print("=" * 55)
print("FINAL CLEANUP")
print("=" * 55)

print(f"\nRows before cleanup : {len(df):,}")

# Step 1: Drop any row with NA in any column
df.dropna(subset=["prev", "curr", "n"], inplace=True)
print(f"After dropping NAs  : {len(df):,}")

# Step 2: Keep only rows where n is a valid positive integer string
mask = df["n"].astype(str).str.strip().str.isdigit()
df = df[mask].copy()
print(f"After digit filter  : {len(df):,}")

# Step 3: Convert types
df["n"] = df["n"].astype(str).str.strip().astype("int32")
df["type"] = df["type"].astype("category")
df.reset_index(drop=True, inplace=True)

# Step 4: Report
mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)

print(f"\nFinal shape  : {len(df):,} rows × {df.shape[1]} columns")
print(f"Memory usage : {mem_mb:.1f} MB")
print(f"\nDtypes:")
print(df.dtypes.to_string())
print(f"\nClick count statistics:")
print(df["n"].describe().apply(lambda x: f"{x:,.0f}").to_string())
print(f"\nSample (first 5 rows):")
print(df.head().to_string(index=False))
print(f"\n✓ Clean DataFrame finalized.")

FINAL CLEANUP

Rows before cleanup : 22,273,328
After dropping NAs  : 22,273,239
After digit filter  : 22,273,239

Final shape  : 22,273,239 rows × 4 columns
Memory usage : 3087.9 MB

Dtypes:
prev      object
curr      object
type    category
n          int32

Click count statistics:
count    22,273,239
mean             94
std             798
min              10
25%              14
50%              24
75%              54
max       1,165,759

Sample (first 5 rows):
      prev                  curr type  n
     Bagua Miscellaneous_Symbols link 13
Baguazhang           Gun_(staff) link 13
  Baguette     French_Parliament link 13
  Baguette        Marie_Blachère link 13
  Baguette                  Pâté link 13

✓ Clean DataFrame finalized.


In [7]:
# Cell 15 — Save to Parquet
# Purpose: Persist the clean DataFrame as a compressed Parquet file.
# This becomes our canonical intermediate data file — all future
# phases load from here, never from the raw TSV again.

import pandas as pd
import os
import time

DATA_DIR     = "/content/data"
TARGET_MONTH = "2026-05"
PARQUET_FILE = os.path.join(DATA_DIR, f"clickstream-enwiki-{TARGET_MONTH}.parquet")

print("=" * 55)
print("SAVING TO PARQUET")
print("=" * 55)
print(f"\nOutput: {PARQUET_FILE}")
print(f"Rows to write: {len(df):,}\n")

start = time.time()

df.to_parquet(
    PARQUET_FILE,
    engine="pyarrow",
    compression="snappy",  # Fast compression, good ratio
    index=False,
)

elapsed = time.time() - start

# Report file size
size_mb = os.path.getsize(PARQUET_FILE) / (1024 * 1024)

print(f"✓ Parquet written in {elapsed:.1f} seconds")
print(f"\nFile size comparison:")
print(f"  Raw .gz (compressed)   : 484.6 MB")
print(f"  Filtered .tsv          : 1047.4 MB")
print(f"  Clean .parquet (snappy): {size_mb:.1f} MB")
print(f"\nCompression ratio vs TSV: {1047.4 / size_mb:.1f}x smaller")

# Verify round-trip: reload and confirm shape matches
print(f"\nVerifying round-trip load...")
df_verify = pd.read_parquet(PARQUET_FILE, engine="pyarrow")
assert df_verify.shape == df.shape, "Shape mismatch after round-trip!"
assert list(df_verify.columns) == list(df.columns), "Column mismatch!"
print(f"✓ Round-trip verified: {df_verify.shape[0]:,} rows × {df_verify.shape[1]} columns")
print(f"\nDtypes preserved:")
print(df_verify.dtypes.to_string())
print(f"\n✓ Parquet file is our canonical data source from here on.")

# Clean up the large TSV to free disk space
tsv_path = os.path.join(DATA_DIR, f"links-only-enwiki-{TARGET_MONTH}.tsv")
os.remove(tsv_path)
print(f"\n🗑  Deleted intermediate TSV to free disk space.")

SAVING TO PARQUET

Output: /content/data/clickstream-enwiki-2026-05.parquet
Rows to write: 22,273,239

✓ Parquet written in 18.1 seconds

File size comparison:
  Raw .gz (compressed)   : 484.6 MB
  Filtered .tsv          : 1047.4 MB
  Clean .parquet (snappy): 595.5 MB

Compression ratio vs TSV: 1.8x smaller

Verifying round-trip load...
✓ Round-trip verified: 22,273,239 rows × 4 columns

Dtypes preserved:
prev      object
curr      object
type    category
n          int32

✓ Parquet file is our canonical data source from here on.

🗑  Deleted intermediate TSV to free disk space.


In [8]:
# Cell 16 — Graph Degree Distribution Analysis
# Purpose: Treat the edge list as a directed graph.
# Compute out-degree (how many articles each article links TO)
# and in-degree (how many articles link TO each article).
# This tells us the shape of the Wikipedia link graph for May 2026.

import pandas as pd
import os

DATA_DIR     = "/content/data"
TARGET_MONTH = "2026-05"
PARQUET_FILE = os.path.join(DATA_DIR, f"clickstream-enwiki-{TARGET_MONTH}.parquet")

print("=" * 55)
print("GRAPH DEGREE DISTRIBUTION ANALYSIS")
print("=" * 55)

# Load from Parquet — this is now our standard starting point
df = pd.read_parquet(PARQUET_FILE, engine="pyarrow")
print(f"✓ Loaded {len(df):,} edges from Parquet\n")

# --- Out-degree: for each article, how many distinct targets ---
out_degree = df.groupby("prev")["curr"].nunique().rename("out_degree")

# --- In-degree: for each article, how many distinct sources ---
in_degree = df.groupby("curr")["prev"].nunique().rename("in_degree")

print(f"Out-degree statistics (articles as sources):")
print("-" * 45)
print(out_degree.describe().apply(lambda x: f"{x:,.1f}").to_string())

print(f"\nIn-degree statistics (articles as targets):")
print("-" * 45)
print(in_degree.describe().apply(lambda x: f"{x:,.1f}").to_string())

# --- Unique article counts ---
unique_sources = df["prev"].nunique()
unique_targets = df["curr"].nunique()
all_articles   = pd.Series(
    pd.concat([df["prev"], df["curr"]]).unique()
).nunique()

print(f"\nUnique articles appearing as source : {unique_sources:>10,}")
print(f"Unique articles appearing as target : {unique_targets:>10,}")
print(f"Total unique articles in graph      : {all_articles:>10,}")

# --- Top 10 most-linked-from articles (highest out-degree) ---
print(f"\nTop 10 articles by out-degree (most outgoing clicks):")
print("-" * 45)
print(out_degree.sort_values(ascending=False).head(10).to_string())

# --- Top 10 most-linked-to articles (highest in-degree) ---
print(f"\nTop 10 articles by in-degree (most incoming clicks):")
print("-" * 45)
print(in_degree.sort_values(ascending=False).head(10).to_string())

GRAPH DEGREE DISTRIBUTION ANALYSIS
✓ Loaded 22,273,239 edges from Parquet

Out-degree statistics (articles as sources):
---------------------------------------------
count    2,158,730.0
mean            10.3
std             24.0
min              1.0
25%              1.0
50%              3.0
75%              9.0
max          2,108.0

In-degree statistics (articles as targets):
---------------------------------------------
count    3,196,033.0
mean             7.0
std             20.5
min              1.0
25%              1.0
50%              2.0
75%              6.0
max          5,213.0

Unique articles appearing as source :  2,158,730
Unique articles appearing as target :  3,196,033
Total unique articles in graph      :  3,339,555

Top 10 articles by out-degree (most outgoing clicks):
---------------------------------------------
prev
List_of_American_films_of_2026                                   2108
2026_FIFA_World_Cup_squads                                       1935
List_of_accid

In [9]:
# Cell 17 — Build Click-Weighted Adjacency Index
# Purpose: Convert the edge DataFrame into a dictionary structure
# that maps each article to its possible next articles and their
# click-weight probabilities. This is the core data structure
# for our random walk path reconstruction.

import pandas as pd
import numpy as np
import os
from collections import defaultdict

DATA_DIR     = "/content/data"
TARGET_MONTH = "2026-05"
PARQUET_FILE = os.path.join(DATA_DIR, f"clickstream-enwiki-{TARGET_MONTH}.parquet")

print("=" * 55)
print("BUILDING CLICK-WEIGHTED ADJACENCY INDEX")
print("=" * 55)

df = pd.read_parquet(PARQUET_FILE, engine="pyarrow")
print(f"✓ Loaded {len(df):,} edges\n")

print("Building adjacency index...")
print("Structure: article → [(target, probability), ...]\n")

# Group by source article
# For each source, compute probability of clicking each target
# proportional to click count n

adjacency = {}  # prev → (list of curr, list of probabilities)

grouped = df.groupby("prev", sort=False)

for prev_article, group in grouped:
    targets = group["curr"].tolist()
    counts  = group["n"].to_numpy(dtype=np.float32)
    probs   = counts / counts.sum()  # normalize to probabilities
    adjacency[prev_article] = (targets, probs)

print(f"✓ Adjacency index built.")
print(f"  Total source articles indexed : {len(adjacency):,}")

# Spot-check a well-known article
test_article = "Python_(programming_language)"
if test_article in adjacency:
    targets, probs = adjacency[test_article]
    print(f"\nSpot-check — '{test_article}':")
    print(f"  Outgoing links : {len(targets)}")
    print(f"  Top 5 by click probability:")
    top_idx = np.argsort(probs)[::-1][:5]
    for i in top_idx:
        print(f"    {probs[i]:.3f}  →  {targets[i]}")
else:
    print(f"\n'{test_article}' not found, trying 'Mathematics'...")
    test_article = "Mathematics"
    if test_article in adjacency:
        targets, probs = adjacency[test_article]
        top_idx = np.argsort(probs)[::-1][:5]
        print(f"  Top 5 outgoing from '{test_article}':")
        for i in top_idx:
            print(f"    {probs[i]:.3f}  →  {targets[i]}")

# Memory estimate
import sys
print(f"\nAdjacency index size in memory: ~{sys.getsizeof(adjacency) / (1024*1024):.1f} MB (shallow)")
print(f"\n✓ Ready for random walk path reconstruction.")

BUILDING CLICK-WEIGHTED ADJACENCY INDEX
✓ Loaded 22,273,239 edges

Building adjacency index...
Structure: article → [(target, probability), ...]

✓ Adjacency index built.
  Total source articles indexed : 2,158,730

Spot-check — 'Python_(programming_language)':
  Outgoing links : 182
  Top 5 by click probability:
    0.187  →  Guido_van_Rossum
    0.036  →  Python_Software_Foundation
    0.034  →  ABC_(programming_language)
    0.033  →  Type_system
    0.027  →  Garbage_collection_(computer_science)

Adjacency index size in memory: ~58.7 MB (shallow)

✓ Ready for random walk path reconstruction.


In [ ]:
# Cell 18 — Random Walk Path Reconstruction
# Purpose: Implement a click-weighted random walk to simulate
# how a user drifts through Wikipedia. Test with 10 paths.
# Each walk = one "rabbit hole session".

import numpy as np
import pandas as pd
import uuid
import json
from datetime import datetime

# --- Configuration ---
MIN_PATH_LENGTH = 4   # minimum hops to be considered a rabbit hole
MAX_PATH_LENGTH = 20  # maximum hops before we stop the walk
N_TEST_WALKS    = 10  # small test batch first
RANDOM_SEED     = 42

rng = np.random.default_rng(RANDOM_SEED)

# All articles that can be a starting point (must have outgoing links)
all_sources = list(adjacency.keys())

print("=" * 55)
print("RANDOM WALK — TEST BATCH")
print("=" * 55)
print(f"Walk config: min={MIN_PATH_LENGTH} hops, max={MAX_PATH_LENGTH} hops")
print(f"Test batch : {N_TEST_WALKS} walks\n")

def random_walk(start_article, adjacency, rng, max_steps=MAX_PATH_LENGTH):
    """
    Perform a single click-weighted random walk on the Wikipedia
    link graph starting from start_article.

    At each step, the next article is chosen probabilistically,
    weighted by real click counts from the clickstream data.

    Returns a list of article names representing the path taken.
    Stops when: max steps reached, dead end, or loop detected.
    """
    path    = [start_article]
    visited = {start_article}

    for _ in range(max_steps - 1):
        current = path[-1]

        # Dead end — article has no outgoing clicked links
        if current not in adjacency:
            break

        targets, probs = adjacency[current]

        # Filter out already-visited articles to avoid loops
        unvisited_mask = [t not in visited for t in targets]

        if not any(unvisited_mask):
            break  # All neighbors already visited — stop

        # Re-normalize probabilities over unvisited targets only
        filtered_targets = [t for t, m in zip(targets, unvisited_mask) if m]
        filtered_probs   = np.array(
            [p for p, m in zip(probs, unvisited_mask) if m],
            dtype=np.float32
        )
        filtered_probs /= filtered_probs.sum()  # renormalize

        # Choose next article weighted by click probability
        next_article = rng.choice(filtered_targets, p=filtered_probs)
        path.append(next_article)
        visited.add(next_article)

    return path


def build_session_record(path, month):
    """
    Convert a raw path list into a structured session record
    matching our target dataset schema.
    """
    return {
        "session_id"   : str(uuid.uuid4()),
        "month"        : month,
        "start_article": path[0],
        "end_article"  : path[-1],
        "path"         : path,
        "path_length"  : len(path),
        # topic_drift_score will be added in Phase 3 (feature engineering)
    }


# --- Generate test walks ---
records = []
attempts = 0

while len(records) < N_TEST_WALKS:
    attempts += 1
    start = rng.choice(all_sources)
    path  = random_walk(start, adjacency, rng)

    # Only keep paths long enough to be interesting
    if len(path) >= MIN_PATH_LENGTH:
        records.append(build_session_record(path, TARGET_MONTH))

print(f"Generated {len(records)} valid paths in {attempts} attempts\n")
print("=" * 55)

for i, rec in enumerate(records):
    print(f"\n[{i+1}] session_id : {rec['session_id'][:8]}...")
    print(f"     length     : {rec['path_length']} hops")
    print(f"     start      : {rec['start_article']}")
    print(f"     end        : {rec['end_article']}")
    print(f"     path       : {' → '.join(rec['path'])}")

print(f"\n✓ Random walk function validated on {N_TEST_WALKS} test paths.")

RANDOM WALK — TEST BATCH
Walk config: min=4 hops, max=20 hops
Test batch : 10 walks



In [1]:
# Cell 19 — Memory-Efficient Graph Representation
# Purpose: Encode the entire graph as integer arrays.
# Replace string article names with integer IDs.
# Store adjacency as numpy arrays, not Python dicts of strings.
# Then perform random walks entirely in integer space.

import pandas as pd
import numpy as np
import os
import uuid

DATA_DIR      = "/content/data"
TARGET_MONTH  = "2026-05"
PARQUET_FILE  = os.path.join(DATA_DIR, f"clickstream-enwiki-{TARGET_MONTH}.parquet")

MIN_PATH_LENGTH = 4
MAX_PATH_LENGTH = 20
N_TEST_WALKS    = 10
RANDOM_SEED     = 42
rng = np.random.default_rng(RANDOM_SEED)

print("=" * 55)
print("MEMORY-EFFICIENT GRAPH REPRESENTATION")
print("=" * 55)

# --- Step 1: Load and encode articles as integers ---
df = pd.read_parquet(PARQUET_FILE, engine="pyarrow")
print(f"✓ Loaded {len(df):,} edges")

# Build a vocabulary: article name → integer ID
all_articles = pd.concat([df["prev"], df["curr"]]).unique()
article_to_id = {art: i for i, art in enumerate(all_articles)}
id_to_article = np.array(all_articles)  # integer → article name

print(f"✓ Vocabulary built: {len(id_to_article):,} unique articles")

# Encode prev and curr as integer IDs
prev_ids = np.array([article_to_id[a] for a in df["prev"]], dtype=np.int32)
curr_ids = np.array([article_to_id[a] for a in df["curr"]], dtype=np.int32)
counts   = df["n"].to_numpy(dtype=np.int32)

# Free the DataFrame — we no longer need it
del df
import gc; gc.collect()
print(f"✓ DataFrame freed from memory")

# --- Step 2: Build CSR-style adjacency ---
# Sort by prev_id so we can slice with offsets
sort_idx = np.argsort(prev_ids, kind="stable")
prev_ids = prev_ids[sort_idx]
curr_ids = curr_ids[sort_idx]
counts   = counts[sort_idx]

# Find where each source node's edges start and end
unique_prev, edge_start = np.unique(prev_ids, return_index=True)
edge_end = np.append(edge_start[1:], len(prev_ids))

print(f"✓ CSR adjacency built: {len(unique_prev):,} source nodes")

# Report memory
mem_unique  = unique_prev.nbytes / 1e6
mem_curr    = curr_ids.nbytes / 1e6
mem_counts  = counts.nbytes / 1e6
mem_offsets = (edge_start.nbytes + edge_end.nbytes) / 1e6
print(f"\nMemory usage:")
print(f"  unique_prev  : {mem_unique:.1f} MB")
print(f"  curr_ids     : {mem_curr:.1f} MB")
print(f"  counts       : {mem_counts:.1f} MB")
print(f"  offsets      : {mem_offsets:.1f} MB")
print(f"  id_to_article: {id_to_article.nbytes / 1e6:.1f} MB")
print(f"  TOTAL        : {mem_unique+mem_curr+mem_counts+mem_offsets:.1f} MB")

# --- Step 3: Random walk in integer space ---
def random_walk_int(start_id, rng, max_steps=MAX_PATH_LENGTH):
    path    = [start_id]
    visited = {start_id}

    for _ in range(max_steps - 1):
        current = path[-1]

        # Find current node in unique_prev
        idx = np.searchsorted(unique_prev, current)
        if idx >= len(unique_prev) or unique_prev[idx] != current:
            break  # dead end

        s, e = edge_start[idx], edge_end[idx]
        nbr_ids  = curr_ids[s:e]
        nbr_cnts = counts[s:e].astype(np.float32)

        # Filter visited
        mask = np.array([n not in visited for n in nbr_ids])
        if not mask.any():
            break

        nbr_ids  = nbr_ids[mask]
        nbr_cnts = nbr_cnts[mask]
        nbr_probs = nbr_cnts / nbr_cnts.sum()

        next_id = rng.choice(nbr_ids, p=nbr_probs)
        path.append(int(next_id))
        visited.add(int(next_id))

    return path

# --- Step 4: Generate test walks ---
source_node_ids = unique_prev.tolist()
records = []
attempts = 0

while len(records) < N_TEST_WALKS and attempts < 1000:
    attempts += 1
    start_id = int(rng.choice(source_node_ids))
    path_ids = random_walk_int(start_id, rng)

    if len(path_ids) >= MIN_PATH_LENGTH:
        path_names = [id_to_article[i] for i in path_ids]
        records.append({
            "session_id"   : str(uuid.uuid4()),
            "month"        : TARGET_MONTH,
            "start_article": path_names[0],
            "end_article"  : path_names[-1],
            "path"         : path_names,
            "path_length"  : len(path_names),
        })

print(f"\n✓ Generated {len(records)} paths in {attempts} attempts\n")
print("=" * 55)
for i, rec in enumerate(records):
    print(f"\n[{i+1}] length : {rec['path_length']} hops")
    print(f"     start  : {rec['start_article']}")
    print(f"     end    : {rec['end_article']}")
    print(f"     path   : {' → '.join(rec['path'])}")

print(f"\n✓ Memory-efficient graph validated.")

MEMORY-EFFICIENT GRAPH REPRESENTATION
✓ Loaded 22,273,239 edges
✓ Vocabulary built: 3,339,555 unique articles
✓ DataFrame freed from memory
✓ CSR adjacency built: 2,158,730 source nodes

Memory usage:
  unique_prev  : 8.6 MB
  curr_ids     : 89.1 MB
  counts       : 89.1 MB
  offsets      : 34.5 MB
  id_to_article: 26.7 MB
  TOTAL        : 221.4 MB

✓ Generated 10 paths in 14 attempts


[1] length : 20 hops
     start  : INS_Magen
     end    : AIR-2_Genie
     path   : INS_Magen → Sa'ar_6-class_corvette → Surface-to-air_missile → Missile → Intercontinental_ballistic_missile → Range_(aeronautics) → Aircraft → Lockheed_Martin_F-22_Raptor → Boeing_F-47 → B61_nuclear_bomb → List_of_nuclear_weapons → MGR-1_Honest_John → Mark_7_nuclear_bomb → Operation_Teapot → Operation_Wigwam → Project_56_(nuclear_test) → Operation_Redwing → Project_57 → Operation_Plumbbob → AIR-2_Genie

[2] length : 12 hops
     start  : Ariana_Grande:_Excuse_Me,_I_Love_You
     end    : Culpables_(Karol_G_and_Anuel_AA_s

In [2]:
# Cell 20 — Generate 50,000 Rabbit Hole Paths
# Purpose: Scale the validated random walk to produce 50,000 sessions.
# Save incrementally to JSONL so progress is preserved if anything interrupts.
# Report distribution statistics on the generated paths.

import numpy as np
import uuid
import json
import os
import time
from tqdm import tqdm

DATA_DIR      = "/content/data"
TARGET_MONTH  = "2026-05"
JSONL_FILE    = os.path.join(DATA_DIR, f"rabbit_holes_{TARGET_MONTH}.jsonl")

N_WALKS         = 50_000
MIN_PATH_LENGTH = 4
MAX_PATH_LENGTH = 20
RANDOM_SEED     = 2024
rng = np.random.default_rng(RANDOM_SEED)

print("=" * 55)
print(f"GENERATING {N_WALKS:,} RABBIT HOLE PATHS")
print("=" * 55)
print(f"Output : {JSONL_FILE}")
print(f"Min hops: {MIN_PATH_LENGTH}  |  Max hops: {MAX_PATH_LENGTH}\n")

source_node_ids = unique_prev  # numpy array — no list conversion needed

records_written = 0
attempts        = 0
path_lengths    = []
start_time      = time.time()

with open(JSONL_FILE, "w", encoding="utf-8") as f:
    with tqdm(total=N_WALKS, unit=" paths", ncols=70) as pbar:

        while records_written < N_WALKS:
            attempts += 1

            # Sample a random starting article
            start_id = int(rng.choice(source_node_ids))
            path_ids = random_walk_int(start_id, rng)

            if len(path_ids) < MIN_PATH_LENGTH:
                continue

            path_names = [id_to_article[i] for i in path_ids]

            record = {
                "session_id"   : str(uuid.uuid4()),
                "month"        : TARGET_MONTH,
                "start_article": path_names[0],
                "end_article"  : path_names[-1],
                "path"         : path_names,
                "path_length"  : len(path_names),
            }

            f.write(json.dumps(record) + "\n")
            path_lengths.append(len(path_names))
            records_written += 1
            pbar.update(1)

elapsed = time.time() - start_time
path_lengths = np.array(path_lengths)

print(f"\n✓ Done in {elapsed:.1f} seconds")
print(f"  Total attempts      : {attempts:,}")
print(f"  Valid paths written : {records_written:,}")
print(f"  Success rate        : {records_written/attempts*100:.1f}%")

print(f"\nPath length distribution:")
print("-" * 35)
for length in range(MIN_PATH_LENGTH, MAX_PATH_LENGTH + 1):
    count = int((path_lengths == length).sum())
    bar   = "█" * (count // 500)
    print(f"  {length:>2} hops : {count:>6,}  {bar}")
print("-" * 35)
print(f"  Mean   : {path_lengths.mean():.1f} hops")
print(f"  Median : {np.median(path_lengths):.1f} hops")

file_size_mb = os.path.getsize(JSONL_FILE) / (1024 * 1024)
print(f"\nJSONL file size: {file_size_mb:.1f} MB")
print(f"✓ {records_written:,} rabbit hole sessions saved to JSONL.")

GENERATING 50,000 RABBIT HOLE PATHS
Output : /content/data/rabbit_holes_2026-05.jsonl
Min hops: 4  |  Max hops: 20



100%|███████████████████████| 50000/50000 [21:04<00:00, 39.55 paths/s]


✓ Done in 1264.3 seconds
  Total attempts      : 63,577
  Valid paths written : 50,000
  Success rate        : 78.6%

Path length distribution:
-----------------------------------
   4 hops :  3,989  ███████
   5 hops :  3,405  ██████
   6 hops :  2,913  █████
   7 hops :  2,639  █████
   8 hops :  2,434  ████
   9 hops :  2,200  ████
  10 hops :  2,046  ████
  11 hops :  1,858  ███
  12 hops :  1,747  ███
  13 hops :  1,624  ███
  14 hops :  1,444  ██
  15 hops :  1,388  ██
  16 hops :  1,331  ██
  17 hops :  1,279  ██
  18 hops :  1,142  ██
  19 hops :  1,108  ██
  20 hops : 17,453  ██████████████████████████████████
-----------------------------------
  Mean   : 13.3 hops
  Median : 14.0 hops

JSONL file size: 24.6 MB
✓ 50,000 rabbit hole sessions saved to JSONL.


In [3]:
# Cell 21 — JSONL Validation
# Purpose: Load all 50,000 records from JSONL and run structural
# integrity checks. Catch any malformed records before Phase 3.

import json
import os
import numpy as np
import pandas as pd
from collections import Counter

DATA_DIR     = "/content/data"
TARGET_MONTH = "2026-05"
JSONL_FILE   = os.path.join(DATA_DIR, f"rabbit_holes_{TARGET_MONTH}.jsonl")

print("=" * 55)
print("JSONL VALIDATION")
print("=" * 55)
print(f"\nFile: {JSONL_FILE}\n")

REQUIRED_FIELDS = {
    "session_id", "month", "start_article",
    "end_article", "path", "path_length"
}

records      = []
errors       = []

with open(JSONL_FILE, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            errors.append(f"Line {line_num}: empty line")
            continue

        try:
            rec = json.loads(line)
        except json.JSONDecodeError as e:
            errors.append(f"Line {line_num}: JSON decode error — {e}")
            continue

        # Check required fields
        missing = REQUIRED_FIELDS - set(rec.keys())
        if missing:
            errors.append(f"Line {line_num}: missing fields {missing}")
            continue

        # Check path_length matches actual path
        if rec["path_length"] != len(rec["path"]):
            errors.append(f"Line {line_num}: path_length mismatch")
            continue

        # Check start/end match path
        if rec["start_article"] != rec["path"][0]:
            errors.append(f"Line {line_num}: start_article mismatch")
            continue
        if rec["end_article"] != rec["path"][-1]:
            errors.append(f"Line {line_num}: end_article mismatch")
            continue

        # Check for duplicate articles within path (loop detection)
        if len(rec["path"]) != len(set(rec["path"])):
            errors.append(f"Line {line_num}: duplicate articles in path")
            continue

        records.append(rec)

print(f"Records loaded  : {len(records):,}")
print(f"Errors found    : {len(errors)}")
if errors:
    print("  First 5 errors:")
    for e in errors[:5]:
        print(f"    {e}")

# --- Distribution checks ---
path_lengths = np.array([r["path_length"] for r in records])
all_starts   = [r["start_article"] for r in records]
all_ends     = [r["end_article"] for r in records]

print(f"\nPath length stats:")
print(f"  Min    : {path_lengths.min()}")
print(f"  Max    : {path_lengths.max()}")
print(f"  Mean   : {path_lengths.mean():.2f}")
print(f"  Median : {np.median(path_lengths):.1f}")

print(f"\nTop 10 most common START articles:")
for art, cnt in Counter(all_starts).most_common(10):
    print(f"  {cnt:>4}x  {art}")

print(f"\nTop 10 most common END articles:")
for art, cnt in Counter(all_ends).most_common(10):
    print(f"  {cnt:>4}x  {art}")

print(f"\nUnique start articles : {len(set(all_starts)):,}")
print(f"Unique end articles   : {len(set(all_ends)):,}")

print(f"\n✓ Validation complete. {len(records):,} clean records confirmed.")

JSONL VALIDATION

File: /content/data/rabbit_holes_2026-05.jsonl

Records loaded  : 50,000
Errors found    : 0

Path length stats:
  Min    : 4
  Max    : 20
  Mean   : 13.32
  Median : 14.0

Top 10 most common START articles:
     3x  John_Lodge_(musician)
     2x  Annanias_Mathe
     2x  February_1953
     2x  Ben_Davino
     2x  Waverly_B._Woodson_Jr.
     2x  JP-4
     2x  2019–20_FIS_Freestyle_Ski_World_Cup
     2x  Grieg_Hall
     2x  Alice_Wade
     2x  722

Top 10 most common END articles:
    10x  Philosophy
     8x  Proposition
     7x  Philosophy_of_language
     7x  Meaning_(philosophy)
     7x  Ancient_Greek
     7x  Abstraction
     7x  Linguistics
     6x  Rule_of_inference
     6x  Communication
     5x  Meet_Me_in_Bluesland

Unique start articles : 49,346
Unique end articles   : 48,196

✓ Validation complete. 50,000 clean records confirmed.


In [4]:
# Cell 22 — Compute Topic Drift Score via TF-IDF Cosine Distance
# Purpose: Vectorize all unique article titles using TF-IDF.
# For each path, compute cosine distance between start and end article.
# This becomes our topic_drift_score — the core analytical feature.

import json
import os
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

DATA_DIR     = "/content/data"
TARGET_MONTH = "2026-05"
JSONL_FILE   = os.path.join(DATA_DIR, f"rabbit_holes_{TARGET_MONTH}.jsonl")

print("=" * 55)
print("FEATURE ENGINEERING — TOPIC DRIFT SCORE")
print("=" * 55)

# --- Step 1: Load all records ---
records = []
with open(JSONL_FILE, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line.strip()))
print(f"✓ Loaded {len(records):,} records")

# --- Step 2: Collect all unique article names ---
unique_articles = set()
for rec in records:
    for article in rec["path"]:
        unique_articles.add(article)
unique_articles = sorted(unique_articles)
art_to_idx = {art: i for i, art in enumerate(unique_articles)}
print(f"✓ Unique articles in paths: {len(unique_articles):,}")

# --- Step 3: Prepare article title text for TF-IDF ---
# Wikipedia article names use underscores — replace with spaces
# Also strip parenthetical disambiguation e.g. "Python_(language)" → "Python"
import re

def clean_title(title):
    title = title.replace("_", " ")
    title = re.sub(r"\(.*?\)", "", title)  # remove parentheticals
    return title.strip()

corpus = [clean_title(art) for art in unique_articles]
print(f"✓ Corpus prepared: {len(corpus):,} article titles")
print(f"  Example: '{unique_articles[100]}' → '{corpus[100]}'")

# --- Step 4: Fit TF-IDF vectorizer ---
print(f"\nFitting TF-IDF vectorizer...")
vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),   # unigrams + bigrams
    min_df=2,             # ignore terms appearing in only 1 article
    max_features=50_000,  # cap vocabulary size for memory
    sublinear_tf=True,    # log normalization
)
tfidf_matrix = vectorizer.fit_transform(corpus)
print(f"✓ TF-IDF matrix: {tfidf_matrix.shape[0]:,} articles × {tfidf_matrix.shape[1]:,} features")
print(f"  Matrix size: {tfidf_matrix.data.nbytes / 1e6:.1f} MB (sparse)")

# --- Step 5: Compute drift score for each record ---
print(f"\nComputing topic drift scores...")

drift_scores = []

for rec in tqdm(records, ncols=70, unit=" recs"):
    start_art = rec["start_article"]
    end_art   = rec["end_article"]

    start_idx = art_to_idx.get(start_art)
    end_idx   = art_to_idx.get(end_art)

    if start_idx is None or end_idx is None:
        drift_scores.append(None)
        continue

    vec_start = tfidf_matrix[start_idx]
    vec_end   = tfidf_matrix[end_idx]

    sim   = cosine_similarity(vec_start, vec_end)[0][0]
    drift = round(1.0 - float(sim), 4)  # distance = 1 - similarity
    drift_scores.append(drift)

valid_scores = [s for s in drift_scores if s is not None]
scores_arr   = np.array(valid_scores)

print(f"\n✓ Drift scores computed: {len(valid_scores):,} valid")
print(f"\nTopic drift score statistics:")
print(f"  Min    : {scores_arr.min():.4f}")
print(f"  Max    : {scores_arr.max():.4f}")
print(f"  Mean   : {scores_arr.mean():.4f}")
print(f"  Median : {np.median(scores_arr):.4f}")
print(f"  Std    : {scores_arr.std():.4f}")

# Show examples at different drift levels
print(f"\nExample paths at different drift levels:")
sorted_idx = np.argsort(scores_arr)

for label, idx in [("Lowest drift ", sorted_idx[0]),
                   ("Median drift ", sorted_idx[len(sorted_idx)//2]),
                   ("Highest drift", sorted_idx[-1])]:
    rec = records[idx]
    print(f"\n  {label} ({scores_arr[idx]:.4f}):")
    print(f"    {rec['start_article']} → ... → {rec['end_article']}")
    print(f"    Path: {' → '.join(rec['path'][:5])}{'...' if len(rec['path'])>5 else ''}")

FEATURE ENGINEERING — TOPIC DRIFT SCORE
✓ Loaded 50,000 records
✓ Unique articles in paths: 440,496
✓ Corpus prepared: 440,496 article titles
  Example: '+−=÷×_Tour' → '+−=÷× Tour'

Fitting TF-IDF vectorizer...
✓ TF-IDF matrix: 440,496 articles × 50,000 features
  Matrix size: 10.3 MB (sparse)

Computing topic drift scores...


100%|██████████████████████| 50000/50000 [00:42<00:00, 1183.92 recs/s]


✓ Drift scores computed: 50,000 valid

Topic drift score statistics:
  Min    : 0.0000
  Max    : 1.0000
  Mean   : 0.9805
  Median : 1.0000
  Std    : 0.1127

Example paths at different drift levels:

  Lowest drift  (0.0000):
    Prince_Fushimi_Sadanaru → ... → Prince_Fushimi_Hiroyoshi
    Path: Prince_Fushimi_Sadanaru → Prince_Fushimi_Hiroyasu → Hiroaki_Fushimi → Prince_Fushimi_Hiroyoshi

  Median drift  (1.0000):
    Usatove_culture → ... → Modal_haplotype
    Path: Usatove_culture → Cernavodă_culture → Karanovo_culture → Old_Europe_(archaeology) → Danubian_culture...

  Highest drift (1.0000):
    Nick_Giannopoulos → ... → Disease
    Path: Nick_Giannopoulos → George_Kapiniaris → Underbelly_(TV_series) → Fat_Tony_&_Co. → Stephen_Curry_(comedian)...


In [5]:
# Cell 23 — Improved Multi-Signal Topic Drift Features
# Purpose: Replace single start/end comparison with path-level features.
# drift_score      = mean cosine distance between consecutive hops
# max_step_drift   = largest single-hop topic jump in the path
# semantic_spread  = mean distance of all articles from path centroid
# These three together give a rich picture of how a rabbit hole unfolds.

import json
import numpy as np
from sklearn.metrics.pairwise import cosine_distances
from tqdm import tqdm

print("=" * 55)
print("IMPROVED DRIFT FEATURES — PATH-LEVEL SIGNALS")
print("=" * 55)

def compute_path_features(path, art_to_idx, tfidf_matrix):
    """
    Compute three drift features for a single rabbit hole path.

    Returns dict with:
      drift_score     : mean cosine distance between consecutive hops
      max_step_drift  : maximum single-hop cosine distance
      semantic_spread : mean distance from path centroid vector
    """
    # Get TF-IDF vectors for all articles in path
    indices = [art_to_idx.get(art) for art in path]

    # Skip any article not in our vocabulary
    valid = [(i, idx) for i, idx in enumerate(indices) if idx is not None]
    if len(valid) < 2:
        return None

    path_positions = [v[0] for v in valid]
    matrix_indices = [v[1] for v in valid]

    # Stack vectors for this path
    vecs = tfidf_matrix[matrix_indices]  # shape: (n_valid, n_features)

    # --- Feature 1: Mean step-by-step cosine distance ---
    step_distances = []
    for i in range(len(matrix_indices) - 1):
        v1 = tfidf_matrix[matrix_indices[i]]
        v2 = tfidf_matrix[matrix_indices[i + 1]]
        dist = cosine_distances(v1, v2)[0][0]
        step_distances.append(dist)

    drift_score    = round(float(np.mean(step_distances)), 4)
    max_step_drift = round(float(np.max(step_distances)), 4)

    # --- Feature 2: Semantic spread (distance from centroid) ---
    # Centroid = mean of all path vectors
    vecs_dense = np.asarray(vecs.todense())
    centroid   = vecs_dense.mean(axis=0, keepdims=True)
    centroid_norm = centroid / (np.linalg.norm(centroid) + 1e-9)

    spread_distances = []
    for i in range(len(matrix_indices)):
        v = vecs_dense[i:i+1]
        v_norm = v / (np.linalg.norm(v) + 1e-9)
        dist = float(cosine_distances(v_norm, centroid_norm)[0][0])
        spread_distances.append(dist)

    semantic_spread = round(float(np.mean(spread_distances)), 4)

    return {
        "drift_score"    : drift_score,
        "max_step_drift" : max_step_drift,
        "semantic_spread": semantic_spread,
    }

# --- Compute features for all records ---
print(f"\nComputing path-level features for {len(records):,} paths...")

enriched_records = []
failed = 0

for rec in tqdm(records, ncols=70, unit=" recs"):
    features = compute_path_features(rec["path"], art_to_idx, tfidf_matrix)
    if features is None:
        failed += 1
        continue
    enriched = {**rec, **features}
    enriched_records.append(enriched)

print(f"\n✓ Features computed: {len(enriched_records):,} records")
print(f"  Failed (vocab miss): {failed}")

# --- Statistics ---
for feat in ["drift_score", "max_step_drift", "semantic_spread"]:
    vals = np.array([r[feat] for r in enriched_records])
    print(f"\n{feat}:")
    print(f"  Min    : {vals.min():.4f}")
    print(f"  Max    : {vals.max():.4f}")
    print(f"  Mean   : {vals.mean():.4f}")
    print(f"  Median : {np.median(vals):.4f}")
    print(f"  Std    : {vals.std():.4f}")

# --- Show most and least drifty paths ---
drift_vals = np.array([r["drift_score"] for r in enriched_records])
sorted_idx = np.argsort(drift_vals)

print(f"\nLowest drift_score (stayed on topic):")
for idx in sorted_idx[:3]:
    r = enriched_records[idx]
    print(f"  {r['drift_score']:.4f} | {r['start_article']} → {r['end_article']}")

print(f"\nHighest drift_score (true rabbit holes):")
for idx in sorted_idx[-3:]:
    r = enriched_records[idx]
    print(f"  {r['drift_score']:.4f} | {r['start_article']} → {r['end_article']}")

IMPROVED DRIFT FEATURES — PATH-LEVEL SIGNALS

Computing path-level features for 50,000 paths...


100%|████████████████████████| 50000/50000 [31:04<00:00, 26.82 recs/s]



✓ Features computed: 50,000 records
  Failed (vocab miss): 0

drift_score:
  Min    : 0.0000
  Max    : 1.0000
  Mean   : 0.8530
  Median : 0.8906
  Std    : 0.1579

max_step_drift:
  Min    : 0.0000
  Max    : 1.0000
  Mean   : 0.9868
  Median : 1.0000
  Std    : 0.0937

semantic_spread:
  Min    : 0.0000
  Max    : 1.0000
  Mean   : 0.6491
  Median : 0.6897
  Std    : 0.1415

Lowest drift_score (stayed on topic):
  0.0000 | 179th_meridian_east → 175th_meridian_east
  0.0000 | Australian_Ninja_Warrior_season_5 → Australian_Ninja_Warrior_season_1
  0.0000 | Sapindus_drummondii → Sapindus_chrysotrichus

Highest drift_score (true rabbit holes):
  1.0000 | Robert_Weygand → Sulfenamide
  1.0000 | List_of_wedding_guests_of_Prince_Harry_and_Meghan_Markle → Dynamic_psychiatry
  1.0000 | Songs_from_the_Grass_String_Ranch → Meet_Me_in_Bluesland


In [6]:
# Cell 24 — Save Enriched Dataset to Final Parquet
# Purpose: Convert enriched_records to a pandas DataFrame,
# enforce correct dtypes, save as compressed Parquet.
# This is our publishable dataset artifact.

import pandas as pd
import numpy as np
import os
import json

DATA_DIR     = "/content/data"
TARGET_MONTH = "2026-05"
FINAL_PARQUET = os.path.join(DATA_DIR, f"internet_rabbit_holes_{TARGET_MONTH}.parquet")

print("=" * 55)
print("SAVING FINAL ENRICHED DATASET")
print("=" * 55)

# --- Build DataFrame ---
df_final = pd.DataFrame(enriched_records)

# Enforce dtypes
df_final["path_length"]    = df_final["path_length"].astype("int16")
df_final["drift_score"]    = df_final["drift_score"].astype("float32")
df_final["max_step_drift"] = df_final["max_step_drift"].astype("float32")
df_final["semantic_spread"]= df_final["semantic_spread"].astype("float32")
df_final["month"]          = df_final["month"].astype("category")

# path column is a list — keep as object (Parquet handles lists natively)
print(f"✓ DataFrame built: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")

# --- Schema summary ---
print(f"\nFinal dataset schema:")
print("-" * 50)
for col in df_final.columns:
    dtype    = str(df_final[col].dtype)
    n_unique = df_final[col].nunique() if col not in ["path", "session_id"] else "—"
    sample   = df_final[col].iloc[0]
    if col == "path":
        sample = f"[{sample[0]}, ..., {sample[-1]}]"
    elif col == "session_id":
        sample = str(sample)[:18] + "..."
    print(f"  {col:<20} {dtype:<12} unique={n_unique!s:<10} e.g. {sample}")
print("-" * 50)

# --- Memory usage ---
mem_mb = df_final.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"\nDataFrame memory: {mem_mb:.1f} MB")

# --- Save to Parquet ---
df_final.to_parquet(
    FINAL_PARQUET,
    engine="pyarrow",
    compression="snappy",
    index=False,
)
file_mb = os.path.getsize(FINAL_PARQUET) / (1024 * 1024)
print(f"✓ Parquet saved: {file_mb:.1f} MB → {FINAL_PARQUET}")

# --- Round-trip verify ---
df_check = pd.read_parquet(FINAL_PARQUET)
assert df_check.shape == df_final.shape, "Shape mismatch!"
print(f"✓ Round-trip verified: {df_check.shape[0]:,} rows × {df_check.shape[1]} cols")

# --- Quick sanity checks ---
print(f"\nSanity checks:")
print(f"  Null values anywhere      : {df_final.isnull().sum().sum()}")
print(f"  Duplicate session_ids     : {df_final['session_id'].duplicated().sum()}")
print(f"  drift_score range [0,1]   : {df_final['drift_score'].between(0,1).all()}")
print(f"  path_length min/max       : {df_final['path_length'].min()} / {df_final['path_length'].max()}")
print(f"  All paths non-empty       : {(df_final['path'].apply(len) > 0).all()}")

print(f"\n✓ Final dataset ready for GitHub commit and Kaggle publish.")

SAVING FINAL ENRICHED DATASET
✓ DataFrame built: 50,000 rows × 9 columns

Final dataset schema:
--------------------------------------------------
  session_id           object       unique=—          e.g. b8fcb78f-9294-493b...
  month                category     unique=1          e.g. 2026-05
  start_article        object       unique=49346      e.g. UEFA_Euro_2012_Group_C
  end_article          object       unique=48196      e.g. Szymon_Weirauch
  path                 object       unique=—          e.g. [UEFA_Euro_2012_Group_C, ..., Szymon_Weirauch]
  path_length          int16        unique=17         e.g. 4
  drift_score          float32      unique=6198       e.g. 0.7232999801635742
  max_step_drift       float32      unique=1097       e.g. 1.0
  semantic_spread      float32      unique=6302       e.g. 0.4602999985218048
--------------------------------------------------

DataFrame memory: 22.7 MB
✓ Parquet saved: 14.5 MB → /content/data/internet_rabbit_holes_2026-05.parquet
✓ Rou

In [7]:
# Cell 25 — Create GitHub Repository via API
# Purpose: Use the GitHub REST API to create a public repository
# and push our initial folder structure and README.
# No local machine needed — everything from Colab.

import requests
import json
import base64
import os

print("=" * 55)
print("GITHUB REPOSITORY SETUP")
print("=" * 55)

# ----------------------------------------------------------------
# FILL THESE IN BEFORE RUNNING
GITHUB_TOKEN    = "YOUR_GITHUB_PAT_HERE"   # Personal Access Token
GITHUB_USERNAME = "vansh-kumar-007"
REPO_NAME       = "Internet-Rabbit-Hole-Dataset"
# ----------------------------------------------------------------

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept"       : "application/vnd.github.v3+json",
}
API_BASE = "https://api.github.com"

# --- Step 1: Create the repository ---
print(f"\nCreating repository: {GITHUB_USERNAME}/{REPO_NAME}")

repo_payload = {
    "name"       : REPO_NAME,
    "description": "50,000 Wikipedia rabbit hole sessions derived from real human click data. Includes path reconstruction and topic drift scoring.",
    "private"    : False,
    "auto_init"  : False,
}

resp = requests.post(
    f"{API_BASE}/user/repos",
    headers=HEADERS,
    json=repo_payload,
)

if resp.status_code == 201:
    repo_url = resp.json()["html_url"]
    print(f"✓ Repository created: {repo_url}")
elif resp.status_code == 422:
    print(f"⚠ Repository already exists — continuing with existing repo.")
    repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}"
else:
    print(f"✗ Failed: {resp.status_code} — {resp.text}")
    raise SystemExit

# --- Step 2: Push files via GitHub Contents API ---
def push_file(path, content_str, message):
    """Push a single file to the GitHub repository."""
    encoded = base64.b64encode(content_str.encode("utf-8")).decode("utf-8")
    url     = f"{API_BASE}/repos/{GITHUB_USERNAME}/{REPO_NAME}/contents/{path}"

    # Check if file already exists (need SHA to update)
    existing = requests.get(url, headers=HEADERS)
    payload  = {"message": message, "content": encoded}
    if existing.status_code == 200:
        payload["sha"] = existing.json()["sha"]

    resp = requests.put(url, headers=HEADERS, json=payload)
    if resp.status_code in (201, 200):
        print(f"  ✓ {path}")
    else:
        print(f"  ✗ {path} — {resp.status_code}: {resp.text[:100]}")

# --- Step 3: Create README ---
README = f"""# Internet Rabbit Hole Dataset

A dataset of **50,000 Wikipedia rabbit hole sessions** derived from real human
click behavior, showing how users drift across topics while browsing.

## What is a Rabbit Hole?

A rabbit hole session captures a chain of Wikipedia article clicks:
- Starts on one topic (e.g. *Aerodynamics*)
- Drifts through connected articles
- Ends somewhere surprisingly different (e.g. *Roman Empire taxation*)

## Dataset

| Field | Type | Description |
|-------|------|-------------|
| session_id | string | Unique session identifier |
| month | string | Source clickstream month (2026-05) |
| start_article | string | First Wikipedia article |
| end_article | string | Final Wikipedia article |
| path | list | Full sequence of articles visited |
| path_length | int | Number of hops (4–20) |
| drift_score | float | Mean cosine distance between consecutive hops (0–1) |
| max_step_drift | float | Largest single topic jump in path (0–1) |
| semantic_spread | float | Mean distance from path centroid (0–1) |

## Source

Derived from the [Wikimedia Clickstream dataset](https://dumps.wikimedia.org/other/clickstream/)
for English Wikipedia, May 2026. Licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).

## Repository Structure
"""

push_file("README.md", README, "Initial commit: Add README.md")

GITHUB REPOSITORY SETUP

Creating repository: vansh-kumar-007/Internet-Rabbit-Hole-Dataset
✗ Failed: 401 — {
  "message": "Bad credentials",
  "documentation_url": "https://docs.github.com/rest",
  "status": "401"
}


SystemExit: 

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [8]:
# Cell 26 — Verify Repo Contents and Push Missing Files
# Purpose: Check what files currently exist in the repo,
# then push any that are missing.

import requests
import base64

print("=" * 55)
print("REPO VERIFICATION AND COMPLETION")
print("=" * 55)

# --- Reuse your credentials ---
# GITHUB_TOKEN and GITHUB_USERNAME must still be set from Cell 25
# If your session restarted, re-fill them here:
# GITHUB_TOKEN    = "YOUR_GITHUB_PAT_HERE"
# GITHUB_USERNAME = "YOUR_GITHUB_USERNAME"
REPO_NAME = "Internet-Rabbit-Hole-Dataset"

HEADERS  = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept"       : "application/vnd.github.v3+json",
}
API_BASE = "https://api.github.com"
REPO_URL = f"{API_BASE}/repos/{GITHUB_USERNAME}/{REPO_NAME}"

def get_repo_files(path=""):
    """Recursively list all files in the repo."""
    url  = f"{REPO_URL}/contents/{path}"
    resp = requests.get(url, headers=HEADERS)
    if resp.status_code != 200:
        return []
    files = []
    for item in resp.json():
        if item["type"] == "file":
            files.append(item["path"])
        elif item["type"] == "dir":
            files.extend(get_repo_files(item["path"]))
    return files

def push_file(path, content_str, message):
    """Push a single file, updating if it already exists."""
    encoded = base64.b64encode(content_str.encode("utf-8")).decode("utf-8")
    url     = f"{REPO_URL}/contents/{path}"
    existing = requests.get(url, headers=HEADERS)
    payload  = {"message": message, "content": encoded}
    if existing.status_code == 200:
        payload["sha"] = existing.json()["sha"]
    resp = requests.put(url, headers=HEADERS, json=payload)
    status = "✓" if resp.status_code in (200, 201) else f"✗ ({resp.status_code})"
    print(f"  {status}  {path}")

# --- Check current state ---
print("\nCurrent files in repo:")
existing_files = get_repo_files()
for f in existing_files:
    print(f"  • {f}")

# --- Define all required files ---
REQUIREMENTS = """\
requests>=2.32.0
beautifulsoup4>=4.13.0
pandas>=2.2.0
numpy>=2.0.0
pyarrow>=18.0.0
scikit-learn>=1.3.0
tqdm>=4.67.0
"""

GITIGNORE = """\
# Data files — never commit raw data
data/*.gz
data/*.tsv
data/*.parquet
data/*.jsonl
data/*.csv

# Python
__pycache__/
*.py[cod]
.ipynb_checkpoints/
.env
"""

LICENSE = """\
Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0)

This dataset is derived from the Wikimedia Clickstream dataset, which is
licensed under CC BY-SA 4.0.

Full license text: https://creativecommons.org/licenses/by-sa/4.0/legalcode
"""

required_files = {
    "requirements.txt"   : (REQUIREMENTS, "Add requirements.txt"),
    ".gitignore"         : (GITIGNORE,    "Add .gitignore"),
    "LICENSE"            : (LICENSE,      "Add CC BY-SA 4.0 license"),
    "data/.gitkeep"      : ("",           "Add data/ directory"),
    "notebooks/.gitkeep" : ("",           "Add notebooks/ directory"),
    "src/.gitkeep"       : ("",           "Add src/ directory"),
}

# --- Push only missing files ---
missing = {
    path: val for path, val in required_files.items()
    if path not in existing_files
}

if not missing:
    print("\n✓ All required files already present.")
else:
    print(f"\nPushing {len(missing)} missing files:")
    for path, (content, message) in missing.items():
        push_file(path, content, message)

# --- Final state ---
print("\nFinal repo contents:")
final_files = get_repo_files()
for f in final_files:
    print(f"  • {f}")

print(f"\n✓ Repository fully initialized.")
print(f"  https://github.com/{GITHUB_USERNAME}/{REPO_NAME}")

REPO VERIFICATION AND COMPLETION

Current files in repo:

Pushing 6 missing files:
  ✗ (401)  requirements.txt
  ✗ (401)  .gitignore
  ✗ (401)  LICENSE
  ✗ (401)  data/.gitkeep
  ✗ (401)  notebooks/.gitkeep
  ✗ (401)  src/.gitkeep

Final repo contents:

✓ Repository fully initialized.
  https://github.com/vansh-kumar-007/Internet-Rabbit-Hole-Dataset


In [9]:
# Cell 27 — Export and Push Notebook to GitHub
# Purpose: Download the current Colab notebook as .ipynb
# and push it to the notebooks/ directory on GitHub.
# This makes our work permanently visible on the repo.

import requests
import base64
import json
import os

print("=" * 55)
print("PUSHING NOTEBOOK TO GITHUB")
print("=" * 55)

REPO_NAME       = "Internet-Rabbit-Hole-Dataset"
NOTEBOOK_PATH   = "notebooks/01_data_collection.ipynb"

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept"       : "application/vnd.github.v3+json",
}
API_BASE = "https://api.github.com"
REPO_URL = f"{API_BASE}/repos/{GITHUB_USERNAME}/{REPO_NAME}"

# --- Step 1: Get the notebook file from Colab's filesystem ---
# Colab saves the live notebook to /content/ with the session name.
# We find it by looking for .ipynb files in /content/

ipynb_files = [
    f for f in os.listdir("/content/")
    if f.endswith(".ipynb")
]

print(f"\nNotebook files found in /content/:")
for f in ipynb_files:
    size_kb = os.path.getsize(f"/content/{f}") / 1024
    print(f"  {f}  ({size_kb:.1f} KB)")

if not ipynb_files:
    print("\n⚠ No .ipynb found in /content/")
    print("  In Colab: File → Download → Download .ipynb")
    print("  Then re-upload it to /content/ and re-run this cell.")
else:
    # Use the first (or only) notebook found
    notebook_filename = ipynb_files[0]
    notebook_local    = f"/content/{notebook_filename}"

    print(f"\nUsing: {notebook_filename}")

    # --- Step 2: Read and encode the notebook ---
    with open(notebook_local, "rb") as f:
        raw_bytes = f.read()

    encoded = base64.b64encode(raw_bytes).decode("utf-8")
    size_kb = len(raw_bytes) / 1024
    print(f"Notebook size: {size_kb:.1f} KB")

    # --- Step 3: Push to GitHub ---
    url      = f"{REPO_URL}/contents/{NOTEBOOK_PATH}"
    existing = requests.get(url, headers=HEADERS)
    payload  = {
        "message": "Add 01_data_collection notebook (ingestion + path reconstruction + features)",
        "content": encoded,
    }
    if existing.status_code == 200:
        payload["sha"] = existing.json()["sha"]
        print("Updating existing notebook...")
    else:
        print("Creating new notebook file...")

    resp = requests.put(url, headers=HEADERS, json=payload)

    if resp.status_code in (200, 201):
        nb_url = resp.json()["content"]["html_url"]
        print(f"\n✓ Notebook pushed successfully.")
        print(f"  View at: {nb_url}")
    else:
        print(f"\n✗ Push failed: {resp.status_code}")
        print(resp.text[:300])

PUSHING NOTEBOOK TO GITHUB

Notebook files found in /content/:

⚠ No .ipynb found in /content/
  In Colab: File → Download → Download .ipynb
  Then re-upload it to /content/ and re-run this cell.


In [ ]:
# Cell 28 — Mount Drive, Find Notebook, Push to GitHub
# Purpose: Mount Google Drive where Colab stores notebooks,
# locate the .ipynb file, and push it to GitHub.

import os
import base64
import requests
from google.colab import drive

print("=" * 55)
print("MOUNT DRIVE AND PUSH NOTEBOOK TO GITHUB")
print("=" * 55)

# --- Step 1: Mount Google Drive ---
print("\nMounting Google Drive...")
drive.mount("/content/drive", force_remount=False)

# --- Step 2: Find notebook in Drive ---
# Colab notebooks are saved in MyDrive/Colab Notebooks/ by default
SEARCH_DIRS = [
    "/content/drive/MyDrive/Colab Notebooks",
    "/content/drive/MyDrive",
]

print("\nSearching for .ipynb files...")
found_notebooks = []

for search_dir in SEARCH_DIRS:
    if not os.path.exists(search_dir):
        continue
    for fname in os.listdir(search_dir):
        if fname.endswith(".ipynb"):
            full_path = os.path.join(search_dir, fname)
            size_kb   = os.path.getsize(full_path) / 1024
            found_notebooks.append((full_path, fname, size_kb))
            print(f"  Found: {full_path}  ({size_kb:.1f} KB)")

if not found_notebooks:
    print("\n⚠ No notebooks found in standard Colab locations.")
    print("  Please run this to find your notebook manually:")
    print("  !find /content/drive/MyDrive -name '*.ipynb' 2>/dev/null")
else:
    # Pick the largest .ipynb — most likely the active one
    found_notebooks.sort(key=lambda x: x[2], reverse=True)
    notebook_path, notebook_name, size_kb = found_notebooks[0]

    print(f"\nUsing: {notebook_name} ({size_kb:.1f} KB)")

    # --- Step 3: Read and push to GitHub ---
    REPO_NAME     = "Internet-Rabbit-Hole-Dataset"
    GITHUB_PATH   = "notebooks/01_data_collection.ipynb"
    HEADERS = {
        "Authorization": f"token {GITHUB_TOKEN}",
        "Accept"       : "application/vnd.github.v3+json",
    }
    REPO_URL = f"https://api.github.com/repos/{GITHUB_USERNAME}/{REPO_NAME}"

    with open(notebook_path, "rb") as f:
        raw_bytes = f.read()

    encoded  = base64.b64encode(raw_bytes).decode("utf-8")
    url      = f"{REPO_URL}/contents/{GITHUB_PATH}"
    existing = requests.get(url, headers=HEADERS)
    payload  = {
        "message": "Add 01_data_collection notebook",
        "content": encoded,
    }
    if existing.status_code == 200:
        payload["sha"] = existing.json()["sha"]

    resp = requests.put(url, headers=HEADERS, json=payload)

    if resp.status_code in (200, 201):
        nb_url = resp.json()["content"]["html_url"]
        print(f"\n✓ Notebook pushed to GitHub.")
        print(f"  View at: {nb_url}")
    else:
        print(f"\n✗ Push failed: {resp.status_code}: {resp.text[:200]}")

In [ ]:
# Cell 29 — Push Correct Notebook to GitHub
# Purpose: Explicitly target "Internet Rabbit Hole Dataset.ipynb"
# and push it to GitHub, overwriting the incorrect file.

import os
import base64
import requests

print("=" * 55)
print("PUSHING CORRECT NOTEBOOK TO GITHUB")
print("=" * 55)

CORRECT_NOTEBOOK = "/content/drive/MyDrive/Colab Notebooks/Internet Rabbit Hole Dataset.ipynb"
REPO_NAME        = "Internet-Rabbit-Hole-Dataset"
GITHUB_PATH      = "notebooks/01_data_collection.ipynb"

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept"       : "application/vnd.github.v3+json",
}
REPO_URL = f"https://api.github.com/repos/{GITHUB_USERNAME}/{REPO_NAME}"

# Confirm file exists and show size
if not os.path.exists(CORRECT_NOTEBOOK):
    print(f"✗ File not found: {CORRECT_NOTEBOOK}")
else:
    size_kb = os.path.getsize(CORRECT_NOTEBOOK) / 1024
    print(f"\nTarget notebook : Internet Rabbit Hole Dataset.ipynb")
    print(f"Size            : {size_kb:.1f} KB")
    print(f"GitHub path     : {GITHUB_PATH}")

    # Read and encode
    with open(CORRECT_NOTEBOOK, "rb") as f:
        raw_bytes = f.read()
    encoded = base64.b64encode(raw_bytes).decode("utf-8")

    # Get existing file SHA (needed to overwrite)
    url      = f"{REPO_URL}/contents/{GITHUB_PATH}"
    existing = requests.get(url, headers=HEADERS)

    payload = {
        "message": "Add 01_data_collection: ingestion, path reconstruction, drift features",
        "content": encoded,
    }
    if existing.status_code == 200:
        payload["sha"] = existing.json()["sha"]
        print(f"\nOverwriting existing file on GitHub...")
    else:
        print(f"\nCreating new file on GitHub...")

    resp = requests.put(url, headers=HEADERS, json=payload)

    if resp.status_code in (200, 201):
        nb_url = resp.json()["content"]["html_url"]
        print(f"\n✓ Correct notebook pushed successfully.")
        print(f"  View at: {nb_url}")
    else:
        print(f"\n✗ Push failed: {resp.status_code}")
        print(resp.text[:300])

In [ ]:
# Cell 30 — Scrub Token from Notebook and Push Clean Version
# Purpose: Read the notebook JSON, replace any token strings
# with a safe placeholder, then push the sanitized version.
# IMPORTANT: Run this AFTER you have regenerated your PAT.

import json
import re
import base64
import requests
import os

print("=" * 55)
print("SCRUBBING SECRETS AND PUSHING CLEAN NOTEBOOK")
print("=" * 55)

CORRECT_NOTEBOOK = "/content/drive/MyDrive/Colab Notebooks/Internet Rabbit Hole Dataset.ipynb"
REPO_NAME        = "Internet-Rabbit-Hole-Dataset"
GITHUB_PATH      = "notebooks/01_data_collection.ipynb"

# --- Step 1: Read notebook as JSON ---
with open(CORRECT_NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

# --- Step 2: Scrub secrets from all cells ---
# Replace GitHub PAT patterns and any string that looks like a token value
PAT_PATTERNS = [
    # Classic PAT: YOUR_GITHUB_PAT_HERE
    (r'ghp_[A-Za-z0-9]{36}', 'YOUR_GITHUB_PAT_HERE'),
    # Fine-grained PAT: YOUR_GITHUB_PAT_HERE
    (r'github_pat_[A-Za-z0-9_]{82}', 'YOUR_GITHUB_PAT_HERE'),
    # Any assignment to GITHUB_TOKEN with a quoted value
    (r'(GITHUB_TOKEN\s*=\s*)["\'][^"\']+["\']',
     r'\1"YOUR_GITHUB_PAT_HERE"'),
]

scrubbed_count = 0

for cell in nb.get("cells", []):
    if cell.get("cell_type") not in ("code", "raw"):
        continue
    source = cell.get("source", [])
    # source can be a list of strings or a single string
    if isinstance(source, list):
        new_source = []
        for line in source:
            original = line
            for pattern, replacement in PAT_PATTERNS:
                line = re.sub(pattern, replacement, line)
            if line != original:
                scrubbed_count += 1
            new_source.append(line)
        cell["source"] = new_source
    else:
        original = source
        for pattern, replacement in PAT_PATTERNS:
            source = re.sub(pattern, replacement, source)
        if source != original:
            scrubbed_count += 1
        cell["source"] = source

print(f"✓ Scrubbed {scrubbed_count} secret occurrence(s) from notebook")

# --- Step 3: Save scrubbed notebook to /content/ ---
CLEAN_PATH = "/content/01_data_collection_clean.ipynb"
with open(CLEAN_PATH, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)
print(f"✓ Clean notebook saved to {CLEAN_PATH}")

# --- Step 4: Push clean version to GitHub ---
# Make sure you have updated GITHUB_TOKEN to your NEW token before this step
HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept"       : "application/vnd.github.v3+json",
}
REPO_URL = f"https://api.github.com/repos/{GITHUB_USERNAME}/{REPO_NAME}"

with open(CLEAN_PATH, "rb") as f:
    encoded = base64.b64encode(f.read()).decode("utf-8")

url      = f"{REPO_URL}/contents/{GITHUB_PATH}"
existing = requests.get(url, headers=HEADERS)
payload  = {
    "message": "Add 01_data_collection notebook (secrets scrubbed)",
    "content": encoded,
}
if existing.status_code == 200:
    payload["sha"] = existing.json()["sha"]

resp = requests.put(url, headers=HEADERS, json=payload)

if resp.status_code in (200, 201):
    nb_url = resp.json()["content"]["html_url"]
    print(f"\n✓ Clean notebook pushed to GitHub.")
    print(f"  View at: {nb_url}")
else:
    print(f"\n✗ Push failed: {resp.status_code}")
    print(resp.text[:300])

In [ ]:
# Cell 31 — Update Token in Session
# Run this cell FIRST, before anything else.
# Do NOT save this cell to the notebook — delete it after running.

GITHUB_TOKEN = "YOUR_GITHUB_PAT_HERE"  # paste your new PAT here
GITHUB_USERNAME = "vansh-kumar-007"
REPO_NAME = "Internet-Rabbit-Hole-Dataset"

print("✓ Credentials updated in session.")
print(f"  Username : {GITHUB_USERNAME}")
print(f"  Token    : {GITHUB_TOKEN[:4]}{'*' * 20}  (masked)")

In [ ]:
# Cell 32 — Deep Scrub: Source + Outputs + Verify Clean
# Purpose: Scrub secrets from BOTH cell source AND cell outputs,
# then verify no token pattern remains before pushing.

import json
import re
import base64
import requests

print("=" * 55)
print("DEEP SCRUB — SOURCE + OUTPUTS")
print("=" * 55)

CORRECT_NOTEBOOK = "/content/drive/MyDrive/Colab Notebooks/Internet Rabbit Hole Dataset.ipynb"
CLEAN_PATH       = "/content/01_data_collection_clean.ipynb"

with open(CORRECT_NOTEBOOK, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Scrub any string value anywhere in the notebook JSON
TOKEN_PATTERNS = [
    r'ghp_[A-Za-z0-9]{1,}',
    r'github_pat_[A-Za-z0-9_]{1,}',
    r'gho_[A-Za-z0-9]{1,}',
    r'ghu_[A-Za-z0-9]{1,}',
]
REPLACEMENT = "YOUR_GITHUB_PAT_HERE"

def scrub_string(s):
    """Replace any token pattern found in a string."""
    for pattern in TOKEN_PATTERNS:
        s = re.sub(pattern, REPLACEMENT, s)
    return s

def scrub_value(obj, depth=0):
    """Recursively walk the notebook JSON and scrub all strings."""
    if isinstance(obj, str):
        return scrub_string(obj)
    elif isinstance(obj, list):
        return [scrub_value(item, depth+1) for item in obj]
    elif isinstance(obj, dict):
        return {k: scrub_value(v, depth+1) for k, v in obj.items()}
    return obj

nb_clean = scrub_value(nb)

# Save clean version
with open(CLEAN_PATH, "w", encoding="utf-8") as f:
    json.dump(nb_clean, f, indent=1)

# Verify — search raw file text for any remaining token patterns
with open(CLEAN_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

remaining = []
for pattern in TOKEN_PATTERNS:
    matches = re.findall(pattern, raw_text)
    remaining.extend(matches)

if remaining:
    print(f"✗ Still found {len(remaining)} token(s) after scrub:")
    for m in remaining:
        print(f"  {m[:6]}...")
else:
    print(f"✓ Zero token patterns remaining in notebook.")
    print(f"  Clean file: {CLEAN_PATH}")
    print(f"  Size: {len(raw_text)/1024:.1f} KB")

    # Push to GitHub
    HEADERS = {
        "Authorization": f"token {GITHUB_TOKEN}",
        "Accept"       : "application/vnd.github.v3+json",
    }
    REPO_URL = f"https://api.github.com/repos/{GITHUB_USERNAME}/{REPO_NAME}"
    GITHUB_PATH = "notebooks/01_data_collection.ipynb"

    with open(CLEAN_PATH, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    url      = f"{REPO_URL}/contents/{GITHUB_PATH}"
    existing = requests.get(url, headers=HEADERS)
    payload  = {
        "message": "Add 01_data_collection notebook (deep scrub, no secrets)",
        "content": encoded,
    }
    if existing.status_code == 200:
        payload["sha"] = existing.json()["sha"]

    resp = requests.put(url, headers=HEADERS, json=payload)

    if resp.status_code in (200, 201):
        nb_url = resp.json()["content"]["html_url"]
        print(f"\n✓ Clean notebook pushed to GitHub.")
        print(f"  View at: {nb_url}")
    else:
        print(f"\n✗ Push failed: {resp.status_code}")
        print(resp.text[:300])

In [ ]:
# Cell 33 — EDA: Load Dataset and Summary Statistics
# Purpose: Load the final enriched dataset and produce a
# comprehensive statistical summary. This is the opening
# section of our Kaggle EDA notebook.

import pandas as pd
import numpy as np
import os

DATA_DIR      = "/content/data"
TARGET_MONTH  = "2026-05"
PARQUET_FILE  = os.path.join(DATA_DIR, f"internet_rabbit_holes_{TARGET_MONTH}.parquet")

print("=" * 55)
print("INTERNET RABBIT HOLE DATASET — EDA")
print("=" * 55)

df = pd.read_parquet(PARQUET_FILE)
print(f"✓ Loaded {len(df):,} rabbit hole sessions\n")

# --- Basic shape ---
print(f"Dataset shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage  : {df.memory_usage(deep=True).sum()/1e6:.1f} MB")

# --- Column summary ---
print(f"\nColumn summary:")
print("-" * 60)
print(f"{'Column':<20} {'Type':<12} {'Non-null':<12} {'Unique'}")
print("-" * 60)
for col in df.columns:
    dtype    = str(df[col].dtype)
    non_null = df[col].notna().sum()
    unique   = df[col].nunique() if col not in ["path","session_id"] else "—"
    print(f"  {col:<18} {dtype:<12} {non_null:<12,} {unique}")
print("-" * 60)

# --- Numeric feature summary ---
numeric_cols = ["path_length", "drift_score", "max_step_drift", "semantic_spread"]
print(f"\nNumerical feature statistics:")
print(df[numeric_cols].describe().round(4).to_string())

# --- Interesting aggregate facts ---
print(f"\nKey facts about this dataset:")
print(f"  Total articles visited across all paths : {df['path'].apply(len).sum():>12,}")
print(f"  Unique start articles                   : {df['start_article'].nunique():>12,}")
print(f"  Unique end articles                     : {df['end_article'].nunique():>12,}")
print(f"  Paths reaching max length (20 hops)     : {(df['path_length']==20).sum():>12,} ({(df['path_length']==20).mean()*100:.1f}%)")
print(f"  Paths with drift_score > 0.95           : {(df['drift_score']>0.95).sum():>12,} ({(df['drift_score']>0.95).mean()*100:.1f}%)")
print(f"  Paths with drift_score < 0.50           : {(df['drift_score']<0.50).sum():>12,} ({(df['drift_score']<0.50).mean()*100:.1f}%)")

# --- Most common end articles (rabbit hole attractors) ---
print(f"\nTop 15 rabbit hole attractors (most common end articles):")
print("-" * 45)
top_ends = df["end_article"].value_counts().head(15)
for art, cnt in top_ends.items():
    bar = "█" * (cnt // 2)
    print(f"  {cnt:>4}x  {art:<35} {bar}")
print("-" * 45)

print(f"\n✓ Summary statistics complete.")

In [ ]:
# Cell 34 — EDA Visualizations
# Purpose: Generate three publication-quality plots:
# 1. Path length distribution
# 2. Drift score distribution with annotation
# 3. Feature correlation heatmap
# These will be saved as images and embedded in the Kaggle notebook.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

PLOT_DIR = "/content/data/plots"
os.makedirs(PLOT_DIR, exist_ok=True)

# Use a clean style
plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "#f8f8f8",
    "axes.grid"        : True,
    "grid.color"       : "white",
    "grid.linewidth"   : 1.2,
    "font.family"      : "DejaVu Sans",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    "Internet Rabbit Hole Dataset — May 2026\n50,000 Wikipedia Navigation Sessions",
    fontsize=14, fontweight="bold", y=1.02
)

# ── Plot 1: Path Length Distribution ──────────────────────
ax1 = axes[0]
lengths = df["path_length"].values
bins    = np.arange(4, 22) - 0.5
counts, _, patches = ax1.hist(lengths, bins=bins, color="#4C72B0", edgecolor="white", linewidth=0.8)

# Colour the max-length bar differently
patches[-1].set_facecolor("#C44E52")
patches[-1].set_label("Max length (20 hops)")

ax1.set_xlabel("Path Length (hops)", fontsize=11)
ax1.set_ylabel("Number of Sessions", fontsize=11)
ax1.set_title("Path Length Distribution", fontsize=12, fontweight="bold")
ax1.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax1.legend(fontsize=9)
ax1.annotate(f"34.9% hit\nmax length",
             xy=(20, counts[-1]), xytext=(16, counts[-1]*0.85),
             arrowprops=dict(arrowstyle="->", color="#C44E52"),
             color="#C44E52", fontsize=9, fontweight="bold")

# ── Plot 2: Drift Score Distribution ──────────────────────
ax2 = axes[1]
scores = df["drift_score"].values
ax2.hist(scores, bins=50, color="#55A868", edgecolor="white", linewidth=0.5)
ax2.axvline(scores.mean(),   color="#C44E52", linewidth=2,
            linestyle="--", label=f"Mean = {scores.mean():.3f}")
ax2.axvline(np.median(scores), color="#4C72B0", linewidth=2,
            linestyle=":",  label=f"Median = {np.median(scores):.3f}")
ax2.set_xlabel("Drift Score", fontsize=11)
ax2.set_ylabel("Number of Sessions", fontsize=11)
ax2.set_title("Topic Drift Score Distribution", fontsize=12, fontweight="bold")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax2.legend(fontsize=9)

# ── Plot 3: Feature Correlation Heatmap ───────────────────
ax3 = axes[2]
numeric_cols = ["path_length", "drift_score", "max_step_drift", "semantic_spread"]
corr = df[numeric_cols].corr()

labels = ["Path\nLength", "Drift\nScore", "Max Step\nDrift", "Semantic\nSpread"]
im = ax3.imshow(corr.values, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")

ax3.set_xticks(range(len(labels)))
ax3.set_yticks(range(len(labels)))
ax3.set_xticklabels(labels, fontsize=9)
ax3.set_yticklabels(labels, fontsize=9)
ax3.set_title("Feature Correlation Matrix", fontsize=12, fontweight="bold")

for i in range(len(labels)):
    for j in range(len(labels)):
        val = corr.values[i, j]
        ax3.text(j, i, f"{val:.2f}",
                 ha="center", va="center",
                 fontsize=10, fontweight="bold",
                 color="black" if abs(val) < 0.7 else "white")

plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)

plt.tight_layout()

PLOT_PATH = os.path.join(PLOT_DIR, "eda_overview.png")
plt.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()

print(f"\n✓ Plot saved to {PLOT_PATH}")
print(f"  Size: {os.path.getsize(PLOT_PATH)/1024:.1f} KB")

# Print correlation table for reference
print(f"\nCorrelation matrix:")
print(corr.round(3).to_string())

In [ ]:
# Cell 35 — Push EDA Plot to GitHub
# Purpose: Commit the EDA visualization to the repository
# under a new assets/ directory.

import base64
import requests
import os

print("=" * 55)
print("PUSHING EDA PLOT TO GITHUB")
print("=" * 55)

PLOT_PATH   = "/content/data/plots/eda_overview.png"
GITHUB_PATH = "assets/eda_overview.png"
REPO_NAME   = "Internet-Rabbit-Hole-Dataset"

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept"       : "application/vnd.github.v3+json",
}
REPO_URL = f"https://api.github.com/repos/{GITHUB_USERNAME}/{REPO_NAME}"

with open(PLOT_PATH, "rb") as f:
    encoded = base64.b64encode(f.read()).decode("utf-8")

url      = f"{REPO_URL}/contents/{GITHUB_PATH}"
existing = requests.get(url, headers=HEADERS)
payload  = {
    "message": "Add EDA overview visualization (path length, drift score, correlations)",
    "content": encoded,
}
if existing.status_code == 200:
    payload["sha"] = existing.json()["sha"]

resp = requests.put(url, headers=HEADERS, json=payload)

if resp.status_code in (200, 201):
    asset_url = resp.json()["content"]["html_url"]
    raw_url   = resp.json()["content"]["download_url"]
    print(f"✓ Plot pushed to GitHub.")
    print(f"  View at  : {asset_url}")
    print(f"  Raw URL  : {raw_url}")

    # Also update README to embed the plot
    print(f"\nUpdating README to embed visualization...")

    readme_url      = f"{REPO_URL}/contents/README.md"
    readme_existing = requests.get(readme_url, headers=HEADERS)
    readme_sha      = readme_existing.json()["sha"]
    current_readme  = base64.b64decode(
        readme_existing.json()["content"]
    ).decode("utf-8")

    # Append plot embed if not already there
    plot_embed = f"\n## Preview\n\n![EDA Overview]({raw_url})\n"
    if "## Preview" not in current_readme:
        new_readme = current_readme + plot_embed
        encoded_readme = base64.b64encode(
            new_readme.encode("utf-8")
        ).decode("utf-8")
        readme_resp = requests.put(
            readme_url,
            headers=HEADERS,
            json={
                "message": "Add EDA preview plot to README",
                "content": encoded_readme,
                "sha"    : readme_sha,
            }
        )
        if readme_resp.status_code in (200, 201):
            print(f"✓ README updated with plot embed.")
        else:
            print(f"✗ README update failed: {readme_resp.status_code}")
    else:
        print(f"  README already has preview section — skipping.")

else:
    print(f"✗ Push failed: {resp.status_code}")
    print(resp.text[:200])